# Huấn luyện MLP — validation only

Giữ nguyên ID/split nguồn. Data v2 phải có fingerprint preprocessing.
Train và chọn CNN chỉ dùng train/validation. Notebook finalize mới đánh giá test.
Output cũ là baseline lịch sử, không ghi đè. `SMOKE_TEST=True` không phải kết quả đầy đủ.
Gắn dataset/Notebook Output phù hợp và runtime assets (detector + pretrained weights).


In [ ]:
from pathlib import Path
import json, os
SEARCH_ROOT = Path('/kaggle/input')
def input_files(name):
    matches=[]
    for directory,dirs,files in os.walk(SEARCH_ROOT):
        dirs[:]=[d for d in dirs if d not in {'images','__pycache__','.git'}]
        if name in files:matches.append(Path(directory)/name)
    return matches
def find_dataset(derived):
    candidates=[]
    for p in input_files('preparation_report.json'):
        if p.parent.name=='metadata' and ('preprocessing' in json.loads(p.read_text()))==derived:
            candidates.append(p.parent.parent)
    if len(candidates)!=1:raise ValueError(f'Set DATA_ROOT explicitly; candidates: {candidates}')
    return candidates[0]
def find_asset(name):
    matches=input_files(name)
    if len(matches)!=1:raise ValueError(f'Expected one {name}; got {matches}')
    return matches[0]
DATA_ROOT = None
OUTPUT_ROOT = Path('/kaggle/working/hand_gesture_v2')
DEVICE = 'auto'
SMOKE_TEST = False
NUM_WORKERS = 0
WEIGHTS_PATH = None  # MobileNet IMAGENET1K_V1 .pth; no download/fallback during training
import os
if Path('/kaggle').exists():
    os.environ['HG_KAGGLE']='1'
    os.environ['HG_REQUIRE_CUDA']='1'
    os.environ.setdefault('HG_PROGRESS_PATH','/kaggle/working/progress.json')


In [ ]:
import sys, types, os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
for package in ['handgesture','scripts']:
    module=types.ModuleType(package);module.__path__=[];sys.modules[package]=module


In [ ]:
# Shared local source: handgesture/runtime.py
_module = types.ModuleType('handgesture.runtime')
_module.__file__ = 'handgesture/runtime.py'
sys.modules['handgesture.runtime'] = _module
_source = '"""Small shared runtime controls for observable, bounded Kaggle execution."""\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport codecs\nimport json\nimport os\nimport selectors\nimport signal\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=path.name+\'.\', suffix=\'.tmp\', dir=path.parent)\n    try:\n        with os.fdopen(fd, \'w\') as stream:\n            json.dump(value, stream, ensure_ascii=False, allow_nan=False, indent=2)\n            stream.write(\'\\n\')\n        os.replace(temporary, path)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef progress(**fields):\n    """Only actual milestones call this; a watchdog heartbeat is not progress."""\n    path = os.environ.get(\'HG_PROGRESS_PATH\')\n    value = {}\n    if path and Path(path).exists():\n        value = json.loads(Path(path).read_text())\n    value.update(fields)\n    value.update(last_progress_at_utc=utc_now(), pid=os.getpid())\n    if path:\n        atomic_json(path, value)\n    line=json.dumps(value, ensure_ascii=False, allow_nan=False)\n    event_log=os.environ.get(\'HG_EVENT_LOG_PATH\')\n    if event_log and os.environ.get(\'HG_EVENT_LOG_OWNER_PID\')==str(os.getpid()):\n        event_path=Path(event_log);event_path.parent.mkdir(parents=True,exist_ok=True)\n        with event_path.open(\'a\') as stream:stream.write(line+\'\\n\')\n    print(line, flush=True)\n    return value\n\n\n@contextmanager\ndef stage(name, model=None):\n    start = time.monotonic()\n    progress(stage=name, model=model, status=\'running\', completed=0, total=None,\n             epoch=None, error=None, started_at_utc=utc_now())\n    try:\n        yield\n    except BaseException as exc:\n        progress(stage=name, model=model, status=\'timeout\' if isinstance(exc,TimeoutError) else \'error\',\n                 error=f\'{type(exc).__name__}: {exc}\', elapsed_seconds=time.monotonic()-start)\n        raise\n    else:\n        progress(stage=name, model=model, status=\'complete\', elapsed_seconds=time.monotonic()-start)\n\n\ndef validate_workers(workers):\n    if workers < 0:\n        raise ValueError(\'workers must be nonnegative\')\n    if os.environ.get(\'HG_KAGGLE\') == \'1\' and workers:\n        raise ValueError(\'Kaggle runtime requires workers=0; parallel workers are not supported safely\')\n\n\ndef require_cuda():\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is required for Kaggle neural-network training; refusing CPU fallback\')\n    # Test allocation, kernel execution and synchronization, not only driver discovery.\n    x = torch.ones((8,8), device=\'cuda\')\n    result = x @ x\n    torch.cuda.synchronize()\n    if not bool(torch.all(result == 8).item()):\n        raise RuntimeError(\'CUDA compute preflight failed\')\n    progress(gpu=torch.cuda.get_device_name(), cuda=torch.version.cuda)\n\n\ndef _stop_group(process):\n    """Kill descendants too, even if the group leader has already exited."""\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        pass\n    time.sleep(.2)\n    try:\n        os.killpg(process.pid, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    process.wait()\n\n\ndef run_logged(command, log_path, deadline, env=None):\n    """Tee combined output immediately; deadline is absolute monotonic time."""\n    if time.monotonic() >= deadline:\n        raise TimeoutError(\'Experiment deadline exceeded before command start\')\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                               start_new_session=True, env=env)\n    decoder = codecs.getincrementaldecoder(\'utf-8\')(errors=\'replace\')\n    try:\n        with log_path.open(\'a\', buffering=1) as log, selectors.DefaultSelector() as selector:\n            selector.register(process.stdout, selectors.EVENT_READ)\n            while selector.get_map() or process.poll() is None:\n                remaining = deadline-time.monotonic()\n                if remaining <= 0:\n                    raise TimeoutError(\'Experiment deadline exceeded; terminating subprocess group\')\n                for key, _ in selector.select(min(.1,remaining)):\n                    chunk = os.read(key.fileobj.fileno(),65536)\n                    if not chunk:\n                        selector.unregister(key.fileobj)\n                        continue\n                    value = decoder.decode(chunk)\n                    sys.stdout.write(value);sys.stdout.flush()\n                    log.write(value);log.flush()\n            tail = decoder.decode(b\'\',final=True)\n            if tail:\n                sys.stdout.write(tail);sys.stdout.flush();log.write(tail)\n        code = process.wait()\n        if code:\n            raise subprocess.CalledProcessError(code, command)\n    except BaseException:\n        _stop_group(process)\n        raise\n    finally:\n        process.stdout.close()\n'
exec(compile(_source, 'handgesture/runtime.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: handgesture/preprocessing.py
_module = types.ModuleType('handgesture.preprocessing')
_module.__file__ = 'handgesture/preprocessing.py'
sys.modules['handgesture.preprocessing'] = _module
_source = '"""The same geometry is used for Kaggle images and unmirrored webcam frames."""\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\n\nimport numpy as np\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\n@dataclass(frozen=True)\nclass PreprocessingConfig:\n    detector_sha256: str\n    version: str = \'pixel_wrist_max3d_square_v2\'\n    detection_confidence: float = .5\n    presence_confidence: float = .5\n    tracking_confidence: float = .5\n    roi_scale: float = 1.4\n    epsilon: float = 1e-6\n    num_hands: int = 1\n    mirror_input: bool = False\n    detector_input_padding: float = .5\n    detector_runtime_version: str = \'0.10.21\'\n\n    def __post_init__(self):\n        if (self.version != \'pixel_wrist_max3d_square_v2\' or self.roi_scale < 1\n                or self.epsilon <= 0 or self.num_hands != 1 or self.mirror_input\n                or not 0 <= self.detector_input_padding <= 1\n                or any(not 0 <= v <= 1 for v in (self.detection_confidence,\n                       self.presence_confidence, self.tracking_confidence))):\n            raise ValueError(\'Invalid preprocessing configuration\')\n\n    @property\n    def fingerprint(self):\n        return hashlib.sha256(json.dumps(asdict(self), sort_keys=True,\n                                        allow_nan=False).encode()).hexdigest()\n\n    def assert_matches(self, fingerprint):\n        if fingerprint != self.fingerprint:\n            raise ValueError(\'Preprocessing fingerprint mismatch; retrain with this pipeline\')\n\n\ndef normalize_landmarks(landmarks, width, height, epsilon=1e-6):\n    points = np.asarray(landmarks, dtype=np.float64)\n    if points.shape != (21, 3) or not np.isfinite(points).all() or min(width, height) <= 0:\n        raise ValueError(\'Invalid landmarks or image dimensions\')\n    pixel = points * np.array([width, height, width])\n    relative = pixel - pixel[0]\n    scale = np.linalg.norm(relative, axis=1).max()\n    if not np.isfinite(scale) or scale <= epsilon:\n        raise ValueError(\'Degenerate landmark scale\')\n    return (relative / scale).astype(np.float32).reshape(63)\n\n\ndef square_roi(image, landmarks, roi_scale=1.4):\n    height, width = image.shape[:2]\n    points = np.asarray(landmarks, dtype=float)\n    if points.shape != (21, 3) or not np.isfinite(points).all():\n        raise ValueError(\'Invalid ROI landmarks\')\n    xy = points[:, :2] * [width, height]\n    low, high = xy.min(axis=0), xy.max(axis=0)\n    side = int(math.ceil(float((high-low).max()) * roi_scale))\n    if side < 1 or side > 4 * max(width, height):\n        raise ValueError(\'Invalid ROI extent\')\n    center = (low+high)/2\n    left, top = np.floor(center-side/2).astype(int)\n    right, bottom = left+side, top+side\n    sx0, sy0, sx1, sy1 = max(0,left), max(0,top), min(width,right), min(height,bottom)\n    if sx1 <= sx0 or sy1 <= sy0:\n        raise ValueError(\'ROI outside image\')\n    roi = np.zeros((side,side,3), dtype=np.uint8)\n    roi[sy0-top:sy1-top, sx0-left:sx1-left] = image[sy0:sy1,sx0:sx1]\n    return roi, (int(left),int(top),int(right),int(bottom))\n\n\n@dataclass\nclass HandResult:\n    status: str\n    features: object = None\n    roi: object = None\n    landmarks: object = None\n    bbox: object = None\n    reason: str = \'\'\n\n\nclass HandPreprocessor:\n    def __init__(self, config, model_path=None, mode=\'IMAGE\', detector=None):\n        self.config = config\n        self._landmarker = None\n        self._last_timestamp = -1\n        self.mode = mode\n        if detector is not None:\n            self._detect = detector\n            return\n        if mode not in {\'IMAGE\',\'VIDEO\'}:\n            raise ValueError(\'mode must be IMAGE or VIDEO\')\n        if sha256(model_path) != config.detector_sha256:\n            raise ValueError(\'Detector SHA-256 mismatch\')\n        from importlib.metadata import version\n        if version(\'mediapipe\') != config.detector_runtime_version:\n            raise ValueError(\'MediaPipe runtime differs from fingerprint; use the documented v2 environment\')\n        import mediapipe as mp\n        from mediapipe.tasks.python import BaseOptions, vision\n        self._mp = mp\n        options = vision.HandLandmarkerOptions(\n            base_options=BaseOptions(model_asset_path=str(model_path),delegate=BaseOptions.Delegate.CPU),\n            running_mode=getattr(vision.RunningMode,mode), num_hands=config.num_hands,\n            min_hand_detection_confidence=config.detection_confidence,\n            min_hand_presence_confidence=config.presence_confidence,\n            min_tracking_confidence=config.tracking_confidence)\n        self._landmarker = vision.HandLandmarker.create_from_options(options)\n        self._detect = self._mediapipe_detect\n\n    def _mediapipe_detect(self, image, timestamp_ms):\n        value = self._mp.Image(image_format=self._mp.ImageFormat.SRGB,\n                               data=np.ascontiguousarray(image))\n        if self.mode == \'VIDEO\':\n            timestamp_ms = max(int(timestamp_ms), self._last_timestamp+1)\n            self._last_timestamp = timestamp_ms\n            result = self._landmarker.detect_for_video(value,timestamp_ms)\n        else:\n            result = self._landmarker.detect(value)\n        if not result.hand_landmarks:\n            return None\n        return np.array([[p.x,p.y,p.z] for p in result.hand_landmarks[0]])\n\n    def detect(self, image, timestamp_ms=0):\n        if image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:\n            raise ValueError(\'Expected uint8 RGB image HxWx3\')\n        height,width=image.shape[:2]\n        pad_x,pad_y=int(width*self.config.detector_input_padding),int(height*self.config.detector_input_padding)\n        padded=np.pad(image,((pad_y,pad_y),(pad_x,pad_x),(0,0)))\n        points=self._detect(padded,timestamp_ms)\n        return None if points is None else self.map_detector_landmarks(points,width,height)\n\n    def map_detector_landmarks(self,points,width,height):\n        points=np.asarray(points,dtype=float).copy()\n        if points.shape!=(21,3):raise ValueError(\'Invalid detector landmark shape\')\n        pad_x,pad_y=int(width*self.config.detector_input_padding),int(height*self.config.detector_input_padding)\n        points[:,0]=(points[:,0]*(width+2*pad_x)-pad_x)/width\n        points[:,1]=(points[:,1]*(height+2*pad_y)-pad_y)/height\n        points[:,2]*=(width+2*pad_x)/width\n        return points\n\n    def transform(self, image, points):\n        if points is None:\n            return HandResult(\'no_hand\', reason=\'no_hand\')\n        try:\n            features = normalize_landmarks(points,image.shape[1],image.shape[0],self.config.epsilon)\n            roi,bbox = square_roi(image,points,self.config.roi_scale)\n        except ValueError as exc:\n            return HandResult(\'invalid\',reason=str(exc))\n        return HandResult(\'ok\',features,roi,np.asarray(points),bbox)\n\n    def process(self, image, timestamp_ms=0):\n        return self.transform(image,self.detect(image,timestamp_ms))\n\n    def close(self):\n        if self._landmarker is not None:\n            self._landmarker.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self,*args):\n        self.close()\n'
exec(compile(_source, 'handgesture/preprocessing.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: handgesture/data.py
_module = types.ModuleType('handgesture.data')
_module.__file__ = 'handgesture/data.py'
sys.modules['handgesture.data'] = _module
_source = '"""Prepare a separate paired dataset; never reassign the source split."""\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\n\nfrom handgesture.preprocessing import HandPreprocessor, PreprocessingConfig, sha256\nfrom handgesture.runtime import progress, validate_workers\n\nGESTURES = [\'fist\',\'one\',\'peace\',\'three\',\'four\',\'palm\']\nMAPPING = [{\'class_id\': i, \'gesture\': g} for i,g in enumerate(GESTURES)]\nLM_COLUMNS = [f\'lm_{i}\' for i in range(63)]\nSPLITS = [\'train\',\'validation\',\'test\']\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\')\n\n\ndef safe_path(root, relative):\n    root = Path(root).resolve()\n    path = (root/relative).resolve()\n    if not path.is_relative_to(root) or not path.is_file():\n        raise ValueError(f\'Invalid or missing path: {relative}\')\n    return path\n\n\ndef _extract_record(row,source_root,stage,processor):\n    item = {\'sample_id\':row[\'sample_id\'],\'split\':row[\'split\'],\n            \'class_id\':int(row[\'class_id\']),\'gesture\':row[\'gesture\']}\n    path = safe_path(source_root,row[\'image_path\'])\n    if row.get(\'image_sha256\') and sha256(path) != row[\'image_sha256\']:\n        raise ValueError(f\'Source image hash mismatch: {path}\')\n    try:\n        with Image.open(path) as image:\n            rgb = np.array(ImageOps.exif_transpose(image).convert(\'RGB\'))\n        result = processor.process(rgb)\n        item[\'status\'] = result.status\n        item[\'reason\'] = result.reason\n        if result.status == \'ok\':\n            # ID-derived names avoid collisions and path traversal.\n            name = __import__(\'hashlib\').sha256(row[\'sample_id\'].encode()).hexdigest()+\'.png\'\n            relative = f"images/{row[\'split\']}/{row[\'gesture\']}/{name}"\n            destination = stage/relative\n            destination.parent.mkdir(parents=True,exist_ok=True)\n            Image.fromarray(result.roi).save(destination)\n            item.update(image_path=relative,image_sha256=sha256(destination),\n                        width=result.roi.shape[1],height=result.roi.shape[0],\n                        source_image_path=row[\'image_path\'],source_image_sha256=sha256(path),\n                        source_label=row.get(\'source_label\',\'train_val_\'+row[\'gesture\']),\n                        source_filepath=row.get(\'source_filepath\',row[\'image_path\']),\n                        roi_bbox=json.dumps(result.bbox),\n                        **dict(zip(LM_COLUMNS,map(float,result.features))))\n    except (OSError,ValueError) as exc:\n        item.update(status=\'invalid\',reason=f\'{type(exc).__name__}: {exc}\')\n    return item\n\n\n_WORKER_CONTEXT = None\n\n\ndef _initialize_worker(config,model_path,source_root,stage,processor):\n    global _WORKER_CONTEXT\n    detector=processor or HandPreprocessor(config,model_path,mode=\'IMAGE\')\n    _WORKER_CONTEXT=(source_root,stage,detector)\n\n\ndef _worker_record(row):\n    return _extract_record(row,*_WORKER_CONTEXT)\n\n\ndef prepare_dataset(source_root, output_root, model_path=None, processor=None, limit=None, workers=0):\n    validate_workers(workers)\n    progress(stage="preprocessing", status="running", completed=0, total=None, model=None)\n    source_root, output_root = Path(source_root).resolve(), Path(output_root).resolve()\n    if source_root == output_root or output_root.is_relative_to(source_root):\n        raise ValueError(\'Derived data must be outside source dataset\')\n    manifest = source_root/\'metadata/manifest.csv\'\n    frame = pd.read_csv(manifest,float_precision=\'round_trip\')\n    if not frame.sample_id.is_unique:\n        raise ValueError(\'Duplicate source sample IDs\')\n    if limit is not None:\n        if limit < 1: raise ValueError(\'limit must be positive\')\n        frame = frame.groupby([\'split\',\'class_id\'],sort=True).head(limit).copy()\n    owned = processor is None\n    config = processor.config if processor else PreprocessingConfig(sha256(model_path))\n    identity = {\'source_manifest_sha256\':sha256(manifest),\n                \'preprocessing_fingerprint\':config.fingerprint,\'limit_per_class_split\':limit}\n    if output_root.exists():\n        existing, report = load_dataset(output_root)\n        if report[\'identity\'] != identity:\n            raise ValueError(\'Output already contains a different dataset\')\n        progress(stage=\'preprocessing\', status=\'complete\', completed=len(frame), total=len(frame), reused=True)\n        return existing, report\n    stage = output_root.with_name(output_root.name+\'.staging\')\n    stage.mkdir(parents=True,exist_ok=True)\n    identity_path = stage/\'identity.json\'\n    if identity_path.exists() and json.loads(identity_path.read_text()) != identity:\n        raise ValueError(\'Staging identity differs; choose another output root\')\n    write_json(identity_path,identity)\n    journal = stage/\'progress.jsonl\'\n    completed = {}\n    if journal.exists():\n        lines=journal.read_bytes().splitlines(keepends=True)\n        valid_bytes=0\n        for index,line in enumerate(lines):\n            try:item=json.loads(line)\n            except (json.JSONDecodeError,UnicodeDecodeError):\n                if index!=len(lines)-1:raise ValueError(\'Corrupt preprocessing journal before last record\')\n                # A crash can interrupt the final append. Retry only that record.\n                with journal.open(\'r+b\') as stream:stream.truncate(valid_bytes)\n                break\n            completed[item[\'sample_id\']] = item\n            valid_bytes+=len(line)\n        if completed and journal.read_bytes()[-1:]!=b\'\\n\':\n            with journal.open(\'ab\') as stream:stream.write(b\'\\n\')\n    if workers < 0:raise ValueError(\'workers must be nonnegative\')\n    pool=None\n    if workers>1:\n        from concurrent.futures import ProcessPoolExecutor\n        from multiprocessing import get_context\n        pool=ProcessPoolExecutor(max_workers=workers,mp_context=get_context(\'fork\'),\n             initializer=_initialize_worker,initargs=(config,model_path,source_root,stage,processor))\n    elif processor is None:\n        processor = HandPreprocessor(config,model_path,mode=\'IMAGE\')\n    pending=[r for r in frame.to_dict(\'records\') if r[\'sample_id\'] not in completed]\n    initial=len(completed)\n    progress(stage="preprocessing", completed=initial, total=len(frame), resumed=bool(initial))\n    try:\n        items=(pool.map(_worker_record,pending,chunksize=8) if pool else\n               (_extract_record(row,source_root,stage,processor) for row in pending))\n        with journal.open(\'a\') as log:\n            for index,item in enumerate(items,initial+1):\n                log.write(json.dumps(item,allow_nan=False)+\'\\n\');log.flush()\n                completed[item[\'sample_id\']] = item\n                if index == initial+1 or index % 100 == 0 or index == len(frame):\n                    progress(stage=\'preprocessing\', completed=index, total=len(frame))\n    finally:\n        if pool is not None:pool.shutdown(wait=True,cancel_futures=True)\n        elif owned:processor.close()\n    records = [completed[s] for s in frame.sample_id]\n    # Cropping may make previously different files identical. Reject all copies\n    # rather than let duplicate crops leak across splits or retain label conflicts.\n    hashes = {}\n    for record in records:\n        if record[\'status\']==\'ok\': hashes.setdefault(record[\'image_sha256\'],[]).append(record)\n    for group in hashes.values():\n        if len(group)>1:\n            for record in group:\n                record.update(status=\'invalid\',reason=\'duplicate_derived_image\')\n                (stage/record[\'image_path\']).unlink(missing_ok=True)\n    accepted = pd.DataFrame([r for r in records if r[\'status\']==\'ok\']).drop(columns=[\'status\',\'reason\'],errors=\'ignore\')\n    rejected = pd.DataFrame([r for r in records if r[\'status\']!=\'ok\'],\n                            columns=[\'sample_id\',\'split\',\'class_id\',\'gesture\',\'status\',\'reason\'])\n    if accepted.empty:\n        raise ValueError(\'No accepted samples; inspect staging progress.jsonl\')\n    metadata = stage/\'metadata\';metadata.mkdir(exist_ok=True)\n    accepted.sort_values(\'sample_id\').to_csv(metadata/\'manifest.csv\',index=False)\n    rejected.to_csv(metadata/\'rejections.csv\',index=False)\n    report = {\'format_version\':2,\'identity\':identity,\'preprocessing\':asdict(config),\n              \'preprocessing_fingerprint\':config.fingerprint,\'class_mapping\':MAPPING,\n              \'smoke_test\':limit is not None,\n              \'counts\':{\'source\':len(frame),\'accepted\':len(accepted),\'rejected\':len(rejected)},\n              \'source_split_counts\':frame.split.value_counts().to_dict(),\n              \'split_counts\':accepted.split.value_counts().to_dict(),\n              \'class_split_counts\':accepted.groupby([\'split\',\'gesture\']).size().unstack(fill_value=0).to_dict(),\n              \'rejection_reasons\':rejected.reason.value_counts().to_dict(),\n              \'artifact_sha256\':{p.name:sha256(p) for p in [metadata/\'manifest.csv\',metadata/\'rejections.csv\']},\n              \'limits\':[\'source random split retained; not subject-wise\',\n                        \'source images were previously cropped; webcam domain differs\']}\n    write_json(metadata/\'preparation_report.json\',report)\n    # Validate complete staged artifact before publishing it.\n    load_dataset(stage)\n    output_root.parent.mkdir(parents=True,exist_ok=True)\n    os.rename(stage,output_root)\n    return load_dataset(output_root)\n\n\ndef load_dataset(root, verify_images=True):\n    root = Path(root)\n    metadata = root/\'metadata\'\n    report = json.loads((metadata/\'preparation_report.json\').read_text())\n    for name,digest in report[\'artifact_sha256\'].items():\n        if sha256(metadata/name) != digest:\n            raise ValueError(f\'Metadata hash mismatch: {name}\')\n    config = PreprocessingConfig(**report[\'preprocessing\'])\n    config.assert_matches(report[\'preprocessing_fingerprint\'])\n    frame = pd.read_csv(metadata/\'manifest.csv\',float_precision=\'round_trip\')\n    required = [\'sample_id\',\'split\',\'class_id\',\'gesture\',\'image_path\',\'image_sha256\',*LM_COLUMNS]\n    if set(required)-set(frame) or frame[required].isna().any().any():\n        raise ValueError(\'Incomplete manifest\')\n    if not np.isfinite(frame[LM_COLUMNS].to_numpy()).all():\n        raise ValueError(\'Nonfinite features\')\n    for col in [\'sample_id\',\'image_path\',\'image_sha256\']:\n        if not frame[col].is_unique: raise ValueError(f\'Duplicate {col}\')\n    if (frame.split.value_counts().to_dict() != report[\'split_counts\']\n            or set(frame.split) != set(SPLITS) or report[\'class_mapping\'] != MAPPING):\n        raise ValueError(\'Split counts or mapping mismatch\')\n    for split in SPLITS:\n        if set(frame.loc[frame.split.eq(split),\'class_id\']) != set(range(6)):\n            raise ValueError(f\'Split {split} is missing classes\')\n    for c,g in enumerate(GESTURES):\n        if not frame.loc[frame.class_id.eq(c),\'gesture\'].eq(g).all():\n            raise ValueError(\'Invalid class mapping\')\n    if not set(frame.class_id).issubset(range(6)):\n        raise ValueError(\'Invalid class ID\')\n    if verify_images:\n        for row in frame.itertuples():\n            if sha256(safe_path(root,row.image_path)) != row.image_sha256:\n                raise ValueError(f\'Image hash mismatch: {row.image_path}\')\n    return frame.sort_values(\'sample_id\').reset_index(drop=True), report\n'
exec(compile(_source, 'handgesture/data.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: scripts/kaggle_notebook_support.py
_module = types.ModuleType('scripts.kaggle_notebook_support')
_module.__file__ = 'scripts/kaggle_notebook_support.py'
sys.modules['scripts.kaggle_notebook_support'] = _module
_source = '"""Source embedded into standalone notebooks by build_kaggle_notebooks.py.\nNo notebook imports this file at runtime.\n"""\n# %% common\nfrom pathlib import Path\nimport copy\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport random\nimport time\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nfrom IPython import get_ipython\nif get_ipython() is None:\n    matplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom PIL import Image, ImageOps, ImageEnhance\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import (accuracy_score, precision_recall_fscore_support,\n                             classification_report, confusion_matrix)\nfrom IPython.display import display, Markdown\n\nGESTURES = [\'fist\', \'one\', \'peace\', \'three\', \'four\', \'palm\']\nCLASS_MAPPING = [{\'class_id\':i, \'gesture\':name} for i,name in enumerate(GESTURES)]\nLM_COLUMNS = [f\'lm_{i}\' for i in range(63)]\nMODEL_ARTIFACTS = {\'rf\':[\'model.joblib\',\'model_config.json\',\'class_mapping.json\'],\n    \'mlp\':[\'model.pt\',\'scaler.joblib\',\'model_config.json\',\'class_mapping.json\'],\n    \'cnn\':[\'model.pt\',\'model_config.json\',\'class_mapping.json\']}\nEXPECTED_SPLITS = {\'train\':27080, \'validation\':5803, \'test\':5804}\nMODEL_ORDER = [\'rf\',\'mlp\',\'cnn\']\nMETRIC_COLUMNS = [\'accuracy\',\'macro_precision\',\'macro_recall\',\'macro_f1\']\nHISTORY_COLUMNS = [\'step\',\'epoch\',\'learning_rate\',\'train_loss\',\'validation_loss\',\n                   \'train_accuracy\',\'validation_accuracy\',\'train_macro_f1\',\'validation_macro_f1\',\'smoke_test\']\nPLOT_STYLE = {\'font.family\':\'DejaVu Sans\',\'font.size\':10,\'axes.titlesize\':12,\n              \'axes.prop_cycle\':plt.cycler(color=[\'#1675cb\',\'#ed8733\',\'#37a36b\'])}\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef find_data_root(explicit=None, search_root=\'/kaggle/input\'):\n    if explicit is not None:\n        root = Path(explicit).expanduser().resolve()\n        if not (root/\'metadata/manifest.csv\').is_file() or not (root/\'images\').is_dir():\n            raise ValueError(f\'DATA_ROOT không chứa metadata/manifest.csv và images/: {root}\')\n        return root\n    candidates = sorted({p.parent.parent for p in Path(search_root).rglob(\'manifest.csv\')\n                         if p.parent.name == \'metadata\' and (p.parent.parent/\'images\').is_dir()})\n    if len(candidates) != 1:\n        raise ValueError(f\'Tìm thấy {len(candidates)} bộ dữ liệu. Đặt DATA_ROOT thủ công. Candidates: {candidates}\')\n    return candidates[0]\n\n\ndef validate_image(root, row):\n    relative = Path(row.image_path)\n    pieces = row.sample_id.split(\'/\', 1)\n    if len(pieces) != 2:\n        raise ValueError(f\'ID mẫu sai: {row.sample_id}\')\n    expected = f\'images/{row.split}/{row.gesture}/{pieces[1]}\'\n    if relative.is_absolute() or \'..\' in relative.parts or row.image_path != expected:\n        raise ValueError(f\'Đường dẫn ảnh sai: {row.image_path}\')\n    path = Path(root)/relative\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    if file_sha256(path) != row.image_sha256:\n        raise ValueError(f\'SHA-256 ảnh không khớp manifest: {row.image_path}\')\n\n\ndef load_dataset(root):\n    root = Path(root)\n    report = json.loads((root/\'metadata/preparation_report.json\').read_text())\n    manifest_path = root/\'metadata/manifest.csv\'\n    digest = file_sha256(manifest_path)\n    if digest != report[\'artifact_sha256\'][\'manifest.csv\']:\n        raise ValueError(\'Hash manifest không khớp preparation_report.json.\')\n    frame = pd.read_csv(manifest_path, float_precision=\'round_trip\')\n    required = [\'sample_id\',\'source_label\',\'class_id\',\'gesture\',\'image_path\',\'image_sha256\',\'split\',*LM_COLUMNS]\n    if set(required)-set(frame.columns) or {col for col in frame.columns if col.startswith(\'lm_\')} != set(LM_COLUMNS):\n        raise ValueError(\'Manifest thiếu cột bắt buộc.\')\n    if frame[required].isna().any().any() or not np.isfinite(frame[LM_COLUMNS].to_numpy(dtype=float)).all():\n        raise ValueError(\'Manifest có dữ liệu thiếu/không hữu hạn.\')\n    if not frame.sample_id.is_unique or not frame.image_sha256.is_unique or not frame.image_path.is_unique:\n        raise ValueError(\'ID/hash/path ảnh trùng, có thể rò rỉ giữa các split.\')\n    if (set(frame[\'split\']) != set(EXPECTED_SPLITS)\n            or frame[\'split\'].value_counts().to_dict() != report[\'split_counts\']):\n        raise ValueError(\'Split không khớp preparation_report.json.\')\n    for mapping in CLASS_MAPPING:\n        class_id, gesture = mapping[\'class_id\'], mapping[\'gesture\']\n        rows = frame.loc[frame.class_id.eq(class_id)]\n        if rows.empty or not rows.gesture.eq(gesture).all() or not rows.source_label.eq(\'train_val_\'+gesture).all():\n            raise ValueError(\'Mapping lớp trong manifest không khớp 6 lớp đã chốt.\')\n        source = report[\'class_mapping\'].get(\'train_val_\'+gesture, {})\n        if source.get(\'class_id\') != class_id or source.get(\'gesture\') != gesture:\n            raise ValueError(\'Mapping trong báo cáo không khớp.\')\n    if set(frame.class_id) != set(range(6)):\n        raise ValueError(\'Manifest có class ID ngoài 0..5.\')\n    for split in EXPECTED_SPLITS:\n        if set(frame.loc[frame.split.eq(split),\'class_id\']) != set(range(6)):\n            raise ValueError(f\'Split {split} thiếu lớp.\')\n    for row in frame[[\'sample_id\',\'split\',\'gesture\',\'image_path\',\'image_sha256\']].itertuples(index=False):\n        validate_image(root, row)\n    return frame.sort_values(\'sample_id\').reset_index(drop=True), report\n\n\ndef smoke_subset(frame, per_class, seed=42):\n    parts=[]\n    for split in [\'train\',\'validation\',\'test\']:\n        for class_id in range(6):\n            rows=frame.loc[frame.split.eq(split)&frame.class_id.eq(class_id)].sort_values(\'sample_id\')\n            if len(rows)<per_class[split]:\n                raise ValueError(\'Không đủ mẫu smoke test theo lớp/split.\')\n            parts.append(rows.sample(per_class[split], random_state=seed+class_id))\n    return pd.concat(parts).sort_values(\'sample_id\').reset_index(drop=True)\n\n\ndef fit_scaler(train_features):\n    scaler=StandardScaler()\n    transformed=scaler.fit_transform(train_features).astype(np.float32)\n    return scaler, transformed\n\n\ndef classification_metrics(y_true, probabilities):\n    predictions=np.asarray(probabilities).argmax(axis=1)\n    precision,recall,f1,_=precision_recall_fscore_support(\n        y_true,predictions,labels=range(6),average=\'macro\',zero_division=0)\n    return {\'accuracy\':float(accuracy_score(y_true,predictions)), \'macro_precision\':float(precision),\n            \'macro_recall\':float(recall),\'macro_f1\':float(f1),\n            \'classification_report\':classification_report(y_true,predictions,labels=range(6),\n                target_names=GESTURES,output_dict=True,zero_division=0)}\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\')\n\n\ndef environment_info(device=\'cpu\'):\n    cpu_name = platform.processor()\n    cpu_file = Path(\'/proc/cpuinfo\')\n    if cpu_file.is_file():\n        cpu_name = next((line.split(\':\',1)[1].strip() for line in cpu_file.read_text().splitlines() if line.startswith(\'model name\')), \'unknown\')\n    info={\'python\':platform.python_version(),\'platform\':platform.platform(),\'cpu\':cpu_name,\n          \'cpu_count\':os.cpu_count(),\'device\':str(device),\n          \'versions\':{p:importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'Pillow\',\'joblib\',\'matplotlib\']}}\n    return info\n\n\ndef protocol_info(frame, source_manifest, smoke, seed):\n    return {\'manifest_sha256\':source_manifest,\'class_mapping\':CLASS_MAPPING,\n            \'smoke_test\':bool(smoke),\'seed\':seed,\n            \'sample_counts\':frame[\'split\'].value_counts().to_dict(),\n            \'test_sample_ids\':frame.loc[frame.split.eq(\'test\'),\'sample_id\'].tolist(),\n            \'limits\':[\'random split does not separate subjects\',\'crop/landmark provenance unverified\',\n                      \'paired classifier evaluation, not webcam end-to-end\']}\n\n\ndef standardize_history(history, model_name, config, smoke=False):\n    """Retain observed values; leave unavailable/non-applicable measurements blank."""\n    history=history.copy()\n    if \'step\' not in history:history[\'step\']=np.arange(1,len(history)+1)\n    if model_name==\'rf\':\n        for column in [\'epoch\',\'learning_rate\',\'train_loss\',\'validation_loss\']:history[column]=np.nan\n    elif \'learning_rate\' not in history and config.get(\'scheduler\') is None:\n        history[\'learning_rate\']=config.get(\'learning_rate\',np.nan)\n    if \'smoke_test\' not in history:history[\'smoke_test\']=bool(smoke)\n    return history.reindex(columns=HISTORY_COLUMNS)\n\n\ndef evaluation_tables(metrics):\n    summary=[];per_class=[]\n    for split,values in [(\'validation\',metrics[\'validation_metrics\']),(\'test\',metrics)]:\n        summary.append({\'model\':metrics[\'model\'],\'split\':split,\n                        \'sample_count\':metrics[\'protocol\'][\'sample_counts\'][split],\n                        **{key:values[key] for key in METRIC_COLUMNS}})\n        for class_id,gesture in enumerate(GESTURES):\n            row=values[\'classification_report\'][gesture]\n            per_class.append({\'model\':metrics[\'model\'],\'split\':split,\'class_id\':class_id,\'gesture\':gesture,\n                              \'precision\':row[\'precision\'],\'recall\':row[\'recall\'],\n                              \'f1_score\':row[\'f1-score\'],\'support\':int(row[\'support\'])})\n    return pd.DataFrame(summary),pd.DataFrame(per_class)\n\n\ndef validate_result_metrics(metrics,predictions):\n    """Check stored test measurements against predictions without replacing them."""\n    measured=classification_metrics(predictions.y_true,predictions[[f\'prob_{i}\' for i in range(6)]].to_numpy())\n    for key in METRIC_COLUMNS:\n        if not np.isclose(metrics[key],measured[key],rtol=1e-9,atol=1e-12):\n            raise ValueError(f"{metrics[\'model\']}: {key} khác CSV dự đoán.")\n    for gesture in GESTURES:\n        for key in [\'precision\',\'recall\',\'f1-score\',\'support\']:\n            if not np.isclose(metrics[\'classification_report\'][gesture][key],\n                              measured[\'classification_report\'][gesture][key],rtol=1e-9,atol=1e-12):\n                raise ValueError(f"{metrics[\'model\']}: {gesture}/{key} khác CSV dự đoán.")\n    for split,values in [(\'validation\',metrics[\'validation_metrics\']),(\'test\',metrics)]:\n        support=sum(values[\'classification_report\'][g][\'support\'] for g in GESTURES)\n        if support!=metrics[\'protocol\'][\'sample_counts\'][split]:\n            raise ValueError(f"{metrics[\'model\']}: support {split} không khớp protocol.")\n\n\ndef finish_figure(fig,path=None):\n    # Fixed canvas: no tight bounding-box cropping, so matching chart types have identical sizes.\n    if path:fig.savefig(path,dpi=130)\n    if get_ipython() is not None and plt.get_backend().lower()!=\'agg\':plt.show()\n    plt.close(fig)\n\n\ndef show_history(history,path,smoke=False,model_name=None):\n    with plt.rc_context(PLOT_STYLE):\n        fig,axes=plt.subplots(1,4,figsize=(18,4),layout=\'constrained\')\n        if smoke:fig.suptitle(\'SMOKE TEST — chỉ kiểm tra pipeline\')\n        if model_name==\'rf\':\n            for index,label in [(0,\'Loss\'),(3,\'Learning rate\')]:\n                axes[index].axis(\'off\');axes[index].text(.5,.5,label+\'\\nKhông áp dụng cho RF\',ha=\'center\',va=\'center\')\n            for ax,metric,label in zip(axes[1:3],[\'accuracy\',\'macro_f1\'],[\'Accuracy\',\'Macro-F1\']):\n                ax.bar([\'train\',\'validation\'],[history[f\'{s}_{metric}\'].iloc[0] for s in [\'train\',\'validation\']],\n                       color=[\'#1675cb\',\'#ed8733\'])\n                ax.set(title=label,ylim=(0,1),xlabel=\'Một lượt fit\')\n        else:\n            for ax,metric,label in zip(axes,[\'loss\',\'accuracy\',\'macro_f1\'],[\'Loss\',\'Accuracy\',\'Macro-F1\']):\n                for split in [\'train\',\'validation\']:\n                    ax.plot(history.epoch,history[f\'{split}_{metric}\'],label=split)\n                ax.set(title=label,xlabel=\'Epoch\');ax.legend()\n                if metric!=\'loss\':ax.set_ylim(0,1)\n            if \'learning_rate\' in history and history.learning_rate.notna().any():\n                axes[3].plot(history.epoch,history.learning_rate)\n                axes[3].set(title=\'Learning rate\',xlabel=\'Epoch\',yscale=\'log\')\n            else:\n                axes[3].axis(\'off\');axes[3].text(.5,.5,\'Learning rate chưa được ghi\',ha=\'center\',va=\'center\')\n        finish_figure(fig,path)\n\n\ndef markdown_table(frame):\n    """Small dependency-free Markdown tables, with blanks rendered as N/A."""\n    def cell(value):\n        if pd.isna(value):return \'N/A\'\n        if isinstance(value,(float,np.floating)):return f\'{value:.6f}\'\n        return str(value).replace(\'|\',\'\\\\|\').replace(\'\\n\',\' \')\n    lines=[\'| \'+\' | \'.join(frame.columns)+\' |\',\'| \'+\' | \'.join([\'---\']*len(frame.columns))+\' |\']\n    lines.extend(\'| \'+\' | \'.join(cell(v) for v in row)+\' |\' for row in frame.itertuples(index=False,name=None))\n    return \'\\n\'.join(lines)\n\n\ndef top_confusions(predictions,model_name,limit=3):\n    cm=confusion_matrix(predictions.y_true,predictions.y_pred,labels=range(6))\n    errors=sorted([(int(cm[i,j]),i,j) for i in range(6) for j in range(6) if i!=j and cm[i,j]>0],\n                  key=lambda x:(-x[0],x[1],x[2]))[:limit]\n    return [{\'model\':model_name,\'true\':GESTURES[i],\'predicted\':GESTURES[j],\'count\':count} for count,i,j in errors]\n\n\ndef write_model_report(output_dir,metrics,predictions):\n    output_dir=Path(output_dir);name=metrics[\'model\'];config=metrics[\'config\'];env=metrics[\'environment\']\n    summary,_=evaluation_tables(metrics)\n    confusions=pd.DataFrame(top_confusions(predictions,name),columns=[\'model\',\'true\',\'predicted\',\'count\'])\n    lines=[f\'# Kết quả {name.upper()}\',\'\']\n    if metrics[\'protocol\'][\'smoke_test\']:lines+=[\'**SMOKE TEST — chỉ kiểm tra pipeline, không dùng để chọn model.**\',\'\']\n    lines+=[\'## Cấu hình\',\'\',f"Phiên: `{config.get(\'run_name\',name)}`; seed: {metrics[\'protocol\'][\'seed\']}.",\n            f"Kiến trúc: {config.get(\'architecture\',\'Random Forest\')}. ",\n            f"Preprocessing: {config.get(\'preprocessing\',\'Không có thông tin\')}."]\n    if name==\'cnn\' and config.get(\'run_name\')==\'cnn_tuned\':lines+=[\'CNN trong báo cáo là phiên tinh chỉnh `cnn_tuned`.\']\n    if name==\'rf\':\n        lines += [f"Số cây: {config.get(\'n_estimators\',\'N/A\')}; max depth: {config.get(\'max_depth\',\'N/A\')}. "]\n    else:\n        lines += [f"Optimizer: {config.get(\'optimizer\',\'N/A\')}; learning rate ban đầu: {config.get(\'learning_rate\',\'N/A\')}; "\n                  f"weight decay: {config.get(\'weight_decay\',\'N/A\')}; batch size: {config.get(\'batch_size\',\'N/A\')}.",\n                  f"Epoch tối đa: {config.get(\'max_epochs\',\'N/A\')}; patience: {config.get(\'patience\',\'N/A\')}; "\n                  f"checkpoint được chọn tại epoch: {config.get(\'best_epoch\',\'N/A\')}.",\n                  \'Scheduler: `\'+json.dumps(config.get(\'scheduler\'),ensure_ascii=False)+\'`.\']\n    lines+=[\'\',\'[Cấu hình đầy đủ](model_config.json)\',\'\', \'## Kết quả\',\'\',markdown_table(summary),\'\',\n            \'[Metric từng lớp](per_class_metrics.csv) · [Metric gốc](metrics.json) · [Dự đoán test](test_predictions.csv)\',\n            \'\', \'## Phân tích nhầm lẫn\',\'\',markdown_table(confusions) if len(confusions) else \'Không có dự đoán sai.\',\n            \'\', \'## Thời gian và thiết bị\',\'\',\n            f"Training: {metrics[\'training_time_s\']:.3f} s; fit wall time: {metrics[\'fit_wall_time_s\']:.3f} s.",\n            f"Device: {env.get(\'device\')}; CPU: {env.get(\'cpu\')}; GPU: {env.get(\'gpu\') or \'N/A\'}.",\n            \'RF đo estimator.fit; mạng đo train batches gồm loader, không gồm validation. \'\n            \'Fit wall time của mạng gồm validation và chọn checkpoint. \'\n            \'Không so sánh tốc độ train công bằng khi phần cứng khác nhau.\',\n            \'\', \'## Biểu đồ\',\'\']\n    for filename,label in [(\'history.png\',\'Lịch sử huấn luyện\'),(\'confusion_matrix.png\',\'Confusion matrix\'),\n                           (\'misclassified_gallery.png\',\'Gallery dự đoán sai\')]:\n        if (output_dir/filename).is_file():lines.append(f\'[{label}]({filename})\')\n    lines+=[\'\',\'[Dữ liệu lịch sử](history.csv)\',\'\',\n            \'RF chỉ có một lượt fit; epoch/loss/learning rate không áp dụng. \'\n            \'Train metrics của MLP/CNN được đo trong batch có dropout/augmentation; validation/test ở eval. \'\n            \'Ô trống là số đo không áp dụng hoặc chưa có, không phải 0. \'\n            \'Test được đánh giá sau khi model/checkpoint cố định. \'\n            \'Split chưa tách người chụp; kết quả không đại diện FPS hoặc webcam end-to-end.\']\n    (output_dir/\'report.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\n\n\ndef write_model_outputs(output_dir,metrics,predictions):\n    """Derive comparable tables and plots; never rewrite measured metrics or model artifacts."""\n    output_dir=Path(output_dir)\n    validate_result_metrics(metrics,predictions)\n    summary,per_class=evaluation_tables(metrics)\n    history=standardize_history(pd.read_csv(output_dir/\'history.csv\',float_precision=\'round_trip\'),\n                                metrics[\'model\'],metrics[\'config\'],metrics[\'protocol\'][\'smoke_test\'])\n    history.to_csv(output_dir/\'history.csv\',index=False)\n    summary.to_csv(output_dir/\'summary.csv\',index=False)\n    per_class.to_csv(output_dir/\'per_class_metrics.csv\',index=False)\n    cm=confusion_matrix(predictions.y_true,predictions.y_pred,labels=range(6))\n    pd.DataFrame(cm,index=GESTURES,columns=GESTURES).to_csv(output_dir/\'confusion_matrix.csv\')\n    pd.DataFrame(cm/np.maximum(cm.sum(axis=1,keepdims=True),1),index=GESTURES,columns=GESTURES).to_csv(output_dir/\'confusion_matrix_normalized.csv\')\n    show_confusion(cm,output_dir/\'confusion_matrix.png\',smoke=metrics[\'protocol\'][\'smoke_test\'])\n    show_history(history,output_dir/\'history.png\',smoke=metrics[\'protocol\'][\'smoke_test\'],model_name=metrics[\'model\'])\n    write_model_report(output_dir,metrics,predictions)\n\n\ndef export_evaluation(output_dir, model_name, frame, probabilities, protocol, env,\n                      training_time, fit_wall_time, validation_metrics, config):\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    metrics=classification_metrics(frame.class_id.to_numpy(),probabilities)\n    artifact_names=MODEL_ARTIFACTS[model_name]\n    metrics.update({\'model\':model_name,\'protocol\':protocol,\'environment\':env,\n                    \'training_time_s\':float(training_time),\'fit_wall_time_s\':float(fit_wall_time),\n                    \'training_time_scope\':\'RF estimator.fit; neural training batches including loader, excluding validation\',\n                    \'validation_metrics\':validation_metrics,\'config\':config,\n                    \'artifact_sha256\':{name:file_sha256(output_dir/name) for name in artifact_names},\n                    \'artifact_bytes\':sum((output_dir/name).stat().st_size for name in artifact_names if (output_dir/name).is_file()),\n                    \'target_accuracy\':.92,\'target_met\':metrics[\'accuracy\']>=.92})\n    result=frame[[\'sample_id\',\'class_id\',\'image_path\']].rename(columns={\'class_id\':\'y_true\'}).copy()\n    result[\'smoke_test\']=protocol[\'smoke_test\']\n    result[\'y_pred\']=np.asarray(probabilities).argmax(axis=1)\n    for class_id in range(6):result[f\'prob_{class_id}\']=probabilities[:,class_id]\n    result.to_csv(output_dir/\'test_predictions.csv\',index=False)\n    metrics[\'test_predictions_sha256\']=file_sha256(output_dir/\'test_predictions.csv\')\n    save_json(output_dir/\'metrics.json\',metrics)\n    write_model_outputs(output_dir,metrics,result)\n    summary,per_class=evaluation_tables(metrics)\n    display(summary.round(4))\n    display(per_class.loc[per_class.split.eq(\'test\')].round(4))\n    display(Markdown(f"**{\'SMOKE TEST — chưa phải huấn luyện đầy đủ. \' if protocol[\'smoke_test\'] else \'\'}"\n                     f"Test Accuracy: {metrics[\'accuracy\']:.2%}; macro-F1: {metrics[\'macro_f1\']:.4f}. "\n                     f"{\'Đạt\' if metrics[\'target_met\'] else \'Chưa đạt\'} mục tiêu 92%.**"))\n    return metrics,result\n\n\ndef show_confusion(cm, path=None, smoke=False):\n    with plt.rc_context(PLOT_STYLE):\n        return _show_confusion(cm,path,smoke)\n\n\ndef _show_confusion(cm, path=None, smoke=False):\n    fig,axes=plt.subplots(1,2,figsize=(12,5),layout=\'constrained\')\n    if smoke:fig.suptitle(\'SMOKE TEST — chỉ kiểm tra pipeline\')\n    normalized=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)\n    for ax,values,title in zip(axes,[cm,normalized],[\'Số lượng\',\'Tỷ lệ theo nhãn thật\']):\n        vmax=1 if values is normalized else max(int(cm.sum(axis=1).max()),1)\n        im=ax.imshow(values,cmap=\'Blues\',vmin=0,vmax=vmax)\n        ax.set_xticks(range(6),GESTURES,rotation=35,ha=\'right\');ax.set_yticks(range(6),GESTURES)\n        ax.set(title=title,xlabel=\'Dự đoán\',ylabel=\'Nhãn thật\')\n        for i in range(6):\n            for j in range(6):\n                label=f\'{values[i,j]:.2f}\' if values is normalized else str(int(values[i,j]))\n                ax.text(j,i,label,ha=\'center\',va=\'center\',color=\'white\' if values[i,j]>vmax/2 else \'black\')\n        fig.colorbar(im,ax=ax,shrink=.8)\n    finish_figure(fig,path)\n\n\ndef show_errors(predictions,data_root,path,limit=12,smoke=False):\n    with plt.rc_context(PLOT_STYLE):\n        return _show_errors(predictions,data_root,path,limit,smoke)\n\n\ndef _show_errors(predictions,data_root,path,limit=12,smoke=False):\n    errors=(predictions.loc[predictions.y_true.ne(predictions.y_pred)].sort_values(\'sample_id\')\n            .groupby(\'y_true\',group_keys=False).head(2).head(limit))\n    if errors.empty:\n        Path(path).unlink(missing_ok=True)\n        print(\'Không có dự đoán sai trong tập được đánh giá.\');return\n    rows=max(3,int(np.ceil(limit/4)))\n    fig,axes=plt.subplots(rows,4,figsize=(12,3*rows),squeeze=False,layout=\'constrained\')\n    if smoke:fig.suptitle(\'SMOKE TEST — chỉ kiểm tra pipeline\')\n    for ax in axes.flat:ax.axis(\'off\')\n    for ax,row in zip(axes.flat,errors.itertuples()):\n        with Image.open(Path(data_root)/row.image_path) as image:\n            ax.imshow(ImageOps.exif_transpose(image).convert(\'RGB\'))\n        ax.set_title(f\'Thật: {GESTURES[row.y_true]}\\nDự đoán: {GESTURES[row.y_pred]}\',fontsize=10)\n    finish_figure(fig,path)\n\n# %% torch\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset,DataLoader,TensorDataset\n\n\ndef seed_everything(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    torch.use_deterministic_algorithms(True)\n\n\ndef select_device(requested=\'auto\',cuda_available=None):\n    available=torch.cuda.is_available() if cuda_available is None else cuda_available\n    if requested==\'auto\':return torch.device(\'cuda\' if available else \'cpu\')\n    if requested==\'cuda\' and not available:raise ValueError(\'Đã yêu cầu CUDA nhưng GPU không khả dụng.\')\n    if requested not in {\'cpu\',\'cuda\'}:raise ValueError(\'DEVICE phải là auto/cpu/cuda.\')\n    return torch.device(requested)\n\n\nclass GestureMLP(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.network=nn.Sequential(nn.Linear(63,128),nn.ReLU(),nn.Dropout(.2),\n                                  nn.Linear(128,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,6))\n    def forward(self,x):return self.network(x)\n\n\nclass GestureCNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        layers=[];channels=3\n        for output in [32,64,128]:\n            layers.extend([nn.Conv2d(channels,output,3,padding=1),nn.BatchNorm2d(output),nn.ReLU(),nn.MaxPool2d(2)])\n            channels=output\n        self.features=nn.Sequential(*layers)\n        self.classifier=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.2),nn.Linear(128,6))\n    def forward(self,x):return self.classifier(self.features(x))\n\n\ndef image_tensor(image,training=False,image_size=128):\n    image=ImageOps.exif_transpose(image).convert(\'RGB\').resize((image_size,image_size),Image.Resampling.BILINEAR)\n    if training:\n        image=image.rotate(random.uniform(-10,10),resample=Image.Resampling.BILINEAR)\n        image=ImageEnhance.Brightness(image).enhance(random.uniform(.85,1.15))\n        image=ImageEnhance.Contrast(image).enhance(random.uniform(.85,1.15))\n    array=np.array(image,dtype=np.float32,copy=True)/255.\n    return torch.from_numpy(array.transpose(2,0,1).copy())\n\n\nclass GestureImages(Dataset):\n    def __init__(self,frame,root,training=False,image_size=128):\n        self.frame=frame.reset_index(drop=True);self.root=Path(root)\n        self.training=training;self.image_size=image_size\n    def __len__(self):return len(self.frame)\n    def __getitem__(self,index):\n        row=self.frame.iloc[index]\n        with Image.open(self.root/row.image_path) as image:\n            tensor=image_tensor(image,self.training,self.image_size)\n        return tensor,int(row.class_id)\n\n\ndef seed_worker(worker_id):\n    seed=torch.initial_seed()%2**32\n    random.seed(seed);np.random.seed(seed)\n\n\ndef make_loader(dataset,batch_size,training,seed,workers=0):\n    generator=torch.Generator().manual_seed(seed)\n    return DataLoader(dataset,batch_size=batch_size,shuffle=training,num_workers=workers,\n                      worker_init_fn=seed_worker,generator=generator,drop_last=False)\n\n\ndef checkpoint_is_better(f1,loss,best_f1,best_loss):\n    return f1>best_f1 or (f1==best_f1 and loss<best_loss)\n\n\ndef synchronize(device):\n    if device.type==\'cuda\':torch.cuda.synchronize(device)\n\n\ndef neural_pass(model,loader,device,optimizer=None):\n    training=optimizer is not None;model.train(training)\n    total_loss=0.;probabilities=[];labels=[]\n    with torch.set_grad_enabled(training):\n        for features,target in loader:\n            features=features.to(device);target=target.to(device)\n            if training:optimizer.zero_grad(set_to_none=True)\n            logits=model(features);loss=nn.functional.cross_entropy(logits,target)\n            if training:loss.backward();optimizer.step()\n            total_loss+=float(loss.detach().cpu())*len(target)\n            probabilities.append(logits.detach().softmax(dim=1).cpu().numpy());labels.append(target.cpu().numpy())\n    values=np.concatenate(probabilities);truth=np.concatenate(labels)\n    metrics=classification_metrics(truth,values)\n    return total_loss/len(truth),metrics,values\n\n\ndef fit_neural(model,train_loader,validation_loader,device,epochs,patience,output_dir,\n               optimizer_config=None,scheduler_config=None):\n    model=model.to(device)\n    optimizer_options={\'lr\':1e-3,\'weight_decay\':1e-4}\n    if optimizer_config is not None:optimizer_options.update(optimizer_config)\n    optimizer=torch.optim.AdamW(model.parameters(),**optimizer_options)\n    scheduler=(torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,**scheduler_config)\n               if scheduler_config is not None else None)\n    history=[];best_f1=-1.;best_loss=float(\'inf\');best_state=None;bad_epochs=0;best_epoch=None\n    training_time=0.;synchronize(device);wall_start=time.perf_counter()\n    for epoch in range(1,epochs+1):\n        learning_rate=optimizer.param_groups[0][\'lr\']\n        synchronize(device);start=time.perf_counter()\n        train_loss,train_metrics,_=neural_pass(model,train_loader,device,optimizer)\n        synchronize(device);training_time+=time.perf_counter()-start\n        validation_loss,validation_metrics,_=neural_pass(model,validation_loader,device)\n        f1=validation_metrics[\'macro_f1\']\n        history.append({\'epoch\':epoch,\'learning_rate\':learning_rate,\'train_loss\':train_loss,\'validation_loss\':validation_loss,\n                        \'train_accuracy\':train_metrics[\'accuracy\'],\'validation_accuracy\':validation_metrics[\'accuracy\'],\n                        \'train_macro_f1\':train_metrics[\'macro_f1\'],\'validation_macro_f1\':f1})\n        print(f\'Epoch {epoch:02d}: lr={learning_rate:.2e}, train loss={train_loss:.4f}, validation loss={validation_loss:.4f}, macro-F1={f1:.4f}\')\n        if checkpoint_is_better(f1,validation_loss,best_f1,best_loss):\n            best_f1,best_loss,best_epoch=f1,validation_loss,epoch\n            best_state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}\n            bad_epochs=0\n        else:bad_epochs+=1\n        # Step after validation: the resulting LR is used in the next epoch.\n        if scheduler is not None:scheduler.step(f1)\n        if bad_epochs>=patience:\n            print(\'Early stopping.\');break\n    synchronize(device);wall_time=time.perf_counter()-wall_start\n    model.load_state_dict(best_state);model.eval()\n    torch.save(best_state,Path(output_dir)/\'model.pt\')\n    pd.DataFrame(history).to_csv(Path(output_dir)/\'history.csv\',index=False)\n    return model,pd.DataFrame(history),training_time,wall_time,best_epoch\n\n\n# %% comparison\n\ndef resolve_comparison_paths(results_root=None,result_dirs=None,output_root=None,start=None,\n                             kaggle_input=\'/kaggle/input\',kaggle_output=\'/kaggle/working/hand_gesture\'):\n    """Resolve canonical local artifacts first, then attached Kaggle outputs."""\n    start=Path(start or Path.cwd()).expanduser().resolve()\n    local_root=next((p/\'results\' for p in [start,*start.parents] if (p/\'results\').is_dir()),None)\n    if results_root is not None:local_root=Path(results_root).expanduser().resolve()\n    kaggle_input=Path(kaggle_input).expanduser().resolve()\n    if result_dirs is not None:\n        dirs={name:Path(folder).expanduser().resolve() for name,folder in result_dirs.items()}\n    elif local_root is not None:\n        dirs={name:local_root/name for name in MODEL_ORDER}\n    elif kaggle_input.is_dir():\n        dirs=find_result_dirs(search_root=kaggle_input)\n    else:\n        raise ValueError(\'Không tìm thấy results/ local hoặc input Kaggle. Đặt RESULTS_ROOT hoặc RESULT_DIRS.\')\n    if output_root is not None:\n        output=Path(output_root).expanduser().resolve()\n    elif dirs and all(folder.is_relative_to(kaggle_input) for folder in dirs.values()):\n        output=Path(kaggle_output).expanduser().resolve()\n    else:\n        output=local_root if local_root is not None else start/\'results\'\n    return dirs,output,local_root\n\n\ndef find_comparison_data_root(explicit=None,start=None,search_root=\'/kaggle/input\'):\n    """Called only when benchmarking: dataset is optional for saved-metric comparison."""\n    if explicit is not None:return find_data_root(explicit)\n    start=Path(start or Path.cwd()).expanduser().resolve()\n    for parent in [start,*start.parents]:\n        candidate=parent/\'data\'\n        if (candidate/\'metadata/manifest.csv\').is_file() and (candidate/\'images\').is_dir():return candidate\n    try:\n        return find_data_root(search_root=search_root)\n    except ValueError as exc:\n        raise ValueError(\'RUN_BENCHMARK=True cần dataset gốc. Đặt DATA_ROOT tới thư mục chứa metadata/ và images/.\') from exc\n\n\ndef run_comparison(result_dirs,output_dir,run_benchmark=False,data_root=None,seed=42,\n                   warmup=20,sample_count=200,data_start=None):\n    """Validate saved predictions and optionally benchmark against the original dataset."""\n    output_dir=Path(output_dir).expanduser().resolve()\n    print(\'Input artifacts:\')\n    for name,folder in result_dirs.items():print(f\'  {name}: {Path(folder).resolve()}\')\n    print(\'Output:\',output_dir)\n    print(\'RUN_BENCHMARK:\',bool(run_benchmark))\n    if output_dir in {Path(p).resolve() for p in result_dirs.values()}:\n        raise ValueError(\'Thư mục comparison phải khác thư mục artifact của model.\')\n    results=load_comparison(result_dirs)\n    reference=results[\'rf\'][\'metrics\'][\'protocol\']\n    if reference.get(\'seed\')!=seed:\n        raise ValueError(f\'Seed/protocol không khớp SEED={seed}.\')\n    for result in results.values():validate_result_metrics(result[\'metrics\'],result[\'predictions\'])\n    latency=None;benchmark_protocol=None\n    if run_benchmark:\n        from threadpoolctl import threadpool_limits\n        root=find_comparison_data_root(data_root,start=data_start)\n        print(\'Benchmark dataset:\',root)\n        full_frame,_=load_dataset(root)\n        if reference[\'manifest_sha256\']!=file_sha256(root/\'metadata/manifest.csv\'):\n            raise ValueError(\'Output không thuộc manifest đang dùng cho benchmark.\')\n        test_frame=full_frame.loc[full_frame.split.eq(\'test\')].set_index(\'sample_id\')\n        test_ids=reference[\'test_sample_ids\']\n        if not set(test_ids).issubset(test_frame.index) or (not reference[\'smoke_test\'] and set(test_ids)!=set(test_frame.index)):\n            raise ValueError(\'Output thiếu ID test hoặc có ID không thuộc test.\')\n        test_frame=test_frame.loc[test_ids].reset_index()\n        for result in results.values():\n            truth=result[\'predictions\'].set_index(\'sample_id\').y_true\n            if not np.array_equal(truth.loc[test_frame.sample_id].to_numpy(),test_frame.class_id.to_numpy()):\n                raise ValueError(\'Nhãn thật output khác manifest.\')\n        with threadpool_limits(limits=1):\n            latency=benchmark_models(results,test_frame,root,seed,warmup,sample_count)\n        benchmark_protocol={\'environment\':environment_info(\'cpu\'),\'torch_version\':torch.__version__,\n                            \'protocol\':reference,\'batch_size\':1,\'threads\':1,\'warmup\':warmup,\n                            \'requested_samples\':sample_count,\n                            \'test_ids\':test_frame.sort_values(\'sample_id\').sample(min(sample_count,len(test_frame)),random_state=seed).sample_id.tolist(),\n                            \'scope\':\'classifier only; MLP includes scaler; CNN precomputed image tensor; no IO/crop/MediaPipe\'}\n    table=export_comparison(results,output_dir,latency=latency)\n    if benchmark_protocol is not None:save_json(output_dir/\'benchmark_protocol.json\',benchmark_protocol)\n    else:(output_dir/\'benchmark_protocol.json\').unlink(missing_ok=True)\n    return table\n\n\ndef load_comparison(result_dirs):\n    if set(result_dirs)!={\'rf\',\'mlp\',\'cnn\'}:raise ValueError(\'Cần đủ rf, mlp, cnn.\')\n    results={};reference=None\n    for name,folder in result_dirs.items():\n        folder=Path(folder)\n        metrics=json.loads((folder/\'metrics.json\').read_text())\n        hashes=metrics.get(\'artifact_sha256\', {})\n        for artifact in MODEL_ARTIFACTS[name]:\n            if artifact not in hashes or not (folder/artifact).is_file() or file_sha256(folder/artifact)!=hashes[artifact]:\n                raise ValueError(f\'Hash artifact không khớp/thiếu: {name}/{artifact}\')\n        if file_sha256(folder/\'test_predictions.csv\')!=metrics.get(\'test_predictions_sha256\'):\n            raise ValueError(f\'Hash prediction CSV không khớp: {name}\')\n        prediction=pd.read_csv(folder/\'test_predictions.csv\')\n        required=[\'sample_id\',\'y_true\',\'y_pred\',*[f\'prob_{i}\' for i in range(6)]]\n        if set(required)-set(prediction.columns) or prediction[required].isna().any().any():\n            raise ValueError(f\'Predictions thiếu/sai: {name}\')\n        if metrics[\'model\']!=name or not prediction.sample_id.is_unique:\n            raise ValueError(\'Model/ID không nhất quán.\')\n        probability=prediction[[f\'prob_{i}\' for i in range(6)]].to_numpy()\n        if (not np.isfinite(probability).all() or (probability<0).any() or (probability>1).any()\n                or not np.allclose(probability.sum(axis=1),1,atol=1e-5)\n                or not np.array_equal(probability.argmax(axis=1),prediction.y_pred.to_numpy())):\n            raise ValueError(\'Xác suất/dự đoán không hợp lệ.\')\n        protocol=metrics[\'protocol\']\n        if protocol[\'class_mapping\'] != CLASS_MAPPING or not set(prediction.y_true).issubset(range(6)):\n            raise ValueError(\'Mapping hoặc nhãn thật không hợp lệ.\')\n        if len(protocol[\'test_sample_ids\']) != len(set(protocol[\'test_sample_ids\'])):\n            raise ValueError(\'Protocol có ID test trùng.\')\n        if set(prediction.sample_id)!=set(protocol[\'test_sample_ids\']):raise ValueError(\'ID test không khớp protocol.\')\n        identity=(protocol[\'manifest_sha256\'],protocol[\'class_mapping\'],protocol[\'smoke_test\'],protocol.get(\'seed\'),\n                  protocol.get(\'sample_counts\'),\n                  prediction.sort_values(\'sample_id\')[[\'sample_id\',\'y_true\']].to_dict(\'records\'))\n        if reference is None:reference=identity\n        elif identity!=reference:raise ValueError(\'Khác mapping, manifest, chế độ smoke hoặc ID/nhãn test; không so sánh.\')\n        results[name]={\'directory\':folder,\'metrics\':metrics,\'predictions\':prediction}\n    return results\n\n\ndef find_result_dirs(explicit=None,search_root=\'/kaggle/input\'):\n    if explicit is not None:return {key:Path(value) for key,value in explicit.items()}\n    found={name:[] for name in [\'rf\',\'mlp\',\'cnn\']}\n    for path in Path(search_root).rglob(\'metrics.json\'):\n        if \'archive\' in path.relative_to(search_root).parts:continue\n        try:\n            obj=json.loads(path.read_text());name=obj.get(\'model\')\n            if name in found and (path.parent/\'test_predictions.csv\').is_file():found[name].append(path.parent)\n        except (OSError,ValueError):continue\n    if any(len(paths)!=1 for paths in found.values()):\n        raise ValueError(f\'Cần một output mỗi model; đặt RESULT_DIRS thủ công: {found}\')\n    return {name:paths[0] for name,paths in found.items()}\n\n\ndef export_comparison(results,output_dir,latency=None):\n    """Summarize verified measurements; latency is included only when actually benchmarked."""\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    rows=[];classes=[];confusions=[]\n    for name in MODEL_ORDER:\n        result=results[name];metrics=result[\'metrics\'];predictions=result[\'predictions\']\n        validate_result_metrics(metrics,predictions)\n        _,per_class=evaluation_tables(metrics);classes.append(per_class)\n        env=metrics[\'environment\'];folder=result[\'directory\']\n        rows.append({\'model\':name,\'run_name\':metrics[\'config\'].get(\'run_name\',name),\n                     \'smoke_test\':metrics[\'protocol\'][\'smoke_test\'],\n                     **{key:metrics[key] for key in METRIC_COLUMNS},\n                     **{f\'validation_{key}\':metrics[\'validation_metrics\'][key] for key in METRIC_COLUMNS},\n                     \'test_samples\':metrics[\'protocol\'][\'sample_counts\'][\'test\'],\n                     \'training_time_s\':metrics[\'training_time_s\'],\'fit_wall_time_s\':metrics[\'fit_wall_time_s\'],\n                     \'training_device\':env.get(\'device\'),\'training_cpu\':env.get(\'cpu\'),\'training_gpu\':env.get(\'gpu\'),\n                     \'artifact_bytes\':sum((folder/f).stat().st_size for f in MODEL_ARTIFACTS[name])})\n        confusions.extend(top_confusions(predictions,name))\n    table=pd.DataFrame(rows);per_class=pd.concat(classes,ignore_index=True)\n    if latency is not None:table=table.merge(latency,on=\'model\',validate=\'one_to_one\')\n    table.to_csv(output_dir/\'comparison.csv\',index=False)\n    per_class.to_csv(output_dir/\'per_class_metrics.csv\',index=False)\n    confusion_table=pd.DataFrame(confusions,columns=[\'model\',\'true\',\'predicted\',\'count\'])\n    confusion_table.to_csv(output_dir/\'top_confusions.csv\',index=False)\n    smoke=bool(table.smoke_test.iloc[0])\n    with plt.rc_context(PLOT_STYLE):\n        fig,axes=plt.subplots(1,3,figsize=(18,4),layout=\'constrained\')\n        if smoke:fig.suptitle(\'SMOKE TEST — chỉ kiểm tra pipeline\')\n        x=np.arange(len(MODEL_ORDER))\n        for ax,metric,label in zip(axes[:2],[\'accuracy\',\'macro_f1\'],[\'Accuracy\',\'Macro-F1\']):\n            for offset,split in [(-.18,\'validation\'),(.18,\'test\')]:\n                values=table[f\'validation_{metric}\'] if split==\'validation\' else table[metric]\n                bars=ax.bar(x+offset,values,width=.36,label=split,\n                            color=\'#ed8733\' if split==\'validation\' else \'#37a36b\')\n                ax.bar_label(bars,fmt=\'%.3f\',fontsize=8,padding=2)\n            ax.set(xticks=x,xticklabels=[n.upper() for n in MODEL_ORDER],ylim=(0,1.1),title=label);ax.legend()\n        for index,name in enumerate(MODEL_ORDER):\n            values=per_class.loc[per_class.model.eq(name)&per_class.split.eq(\'test\'),\'f1_score\']\n            axes[2].plot(range(6),values,marker=\'o\',label=name.upper())\n        axes[2].set(xticks=range(6),xticklabels=GESTURES,ylim=(0,1.05),title=\'Test F1 theo lớp\');axes[2].legend()\n        finish_figure(fig,output_dir/\'comparison.png\')\n        fig,axes=plt.subplots(1,3,figsize=(18,5),layout=\'constrained\')\n        if smoke:fig.suptitle(\'SMOKE TEST — chỉ kiểm tra pipeline\')\n        for ax,name in zip(axes,MODEL_ORDER):\n            p=results[name][\'predictions\'];cm=confusion_matrix(p.y_true,p.y_pred,labels=range(6))\n            normalized=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)\n            im=ax.imshow(normalized,cmap=\'Blues\',vmin=0,vmax=1)\n            ax.set(title=name.upper(),xticks=range(6),xticklabels=GESTURES,\n                   yticks=range(6),yticklabels=GESTURES,xlabel=\'Dự đoán\',ylabel=\'Nhãn thật\')\n            for i in range(6):\n                for j in range(6):\n                    ax.text(j,i,f\'{normalized[i,j]:.2f}\',ha=\'center\',va=\'center\',\n                            color=\'white\' if normalized[i,j]>.5 else \'black\',fontsize=9)\n        fig.colorbar(im,ax=list(axes),shrink=.8)\n        finish_figure(fig,output_dir/\'confusion_matrices.png\')\n    report=[\'# So sánh RF, MLP và CNN\',\'\']\n    if smoke:report+=[\'**SMOKE TEST — không dùng để xếp hạng hoặc chọn model.**\',\'\']\n    report+=[\'Cùng manifest, seed, mapping và tập test. CNN hiện tại là phiên `\'+str(table.loc[table.model.eq(\'cnn\'),\'run_name\'].iloc[0])+\'`.\',\n             \'\', \'## Chất lượng\',\'\',markdown_table(table[[\'model\',\'validation_accuracy\',\'validation_macro_f1\',\n                \'accuracy\',\'macro_f1\',\'test_samples\']]),\'\',\n             \'[Bảng tổng hợp đầy đủ](comparison.csv) · [Metric từng lớp](per_class_metrics.csv)\',\n             \'\', \'## Thời gian và thiết bị\',\'\',markdown_table(table[[\'model\',\'training_time_s\',\'fit_wall_time_s\',\n                \'training_device\',\'training_cpu\',\'training_gpu\']]),\'\',\n             \'Thời gian train mô tả từng phiên; phần cứng khác nhau nên không dùng làm kết luận tốc độ công bằng. \'\n             \'RF/MLP dùng landmark, CNN dùng ảnh. Không tính detector, crop hoặc pipeline webcam.\',\n             \'\', \'## Phân tích lỗi\',\'\',markdown_table(confusion_table),\'\',\n             \'[Biểu đồ chất lượng](comparison.png) · [Confusion matrix cùng thang màu](confusion_matrices.png)\',\n             \'\', \'## Báo cáo từng model\',\'\']\n    for name in MODEL_ORDER:\n        folder=results[name][\'directory\']\n        has_report=(folder/\'report.md\').is_file()\n        target=os.path.relpath(folder/(\'report.md\' if has_report else \'metrics.json\'),output_dir)\n        label=name.upper() if has_report else name.upper()+\' — metric gốc (output cũ chưa có report.md)\'\n        report.append(f\'[{label}]({target})\')\n    if latency is not None:\n        report+=[\'\',\'## Benchmark CPU\',\'\',markdown_table(table[[\'model\',\'cpu_latency_median_ms\',\'cpu_latency_p95_ms\']]),\n                 \'\', \'[Protocol benchmark](benchmark_protocol.json)\',\n                 \'Đây là classifier latency đã đo, không phải FPS toàn hệ thống.\']\n    else:report+=[\'\',\'Chưa chạy benchmark latency trong báo cáo này; không suy ra latency hoặc FPS.\']\n    (output_dir/\'report.md\').write_text(\'\\n\'.join(report)+\'\\n\')\n    return table\n\n\ndef load_saved_model(name,folder):\n    folder=Path(folder);config=json.loads((folder/\'model_config.json\').read_text())\n    if config.get(\'model\')!=name or config.get(\'class_mapping\')!=CLASS_MAPPING:\n        raise ValueError(\'Config model/mapping không khớp baseline.\')\n    if name in {\'rf\',\'mlp\'} and config.get(\'features\')!=LM_COLUMNS:\n        raise ValueError(\'Thứ tự features không khớp baseline.\')\n    if name==\'mlp\' and (config.get(\'architecture\')!=[63,128,64,6] or config.get(\'activation\')!=\'ReLU\' or config.get(\'dropout\')!=.2\n            or config.get(\'preprocessing\')!=\'StandardScaler fit chỉ train; scaler.joblib\'):\n        raise ValueError(\'Kiến trúc/preprocessing MLP không khớp baseline.\')\n    if name==\'cnn\' and (config.get(\'architecture\')!=\'3 x Conv3x3(padding1)-BN-ReLU-MaxPool2, channels32/64/128; GAP-Dropout-Linear6\'\n            or config.get(\'dropout\')!=.2 or config.get(\'image_size\')!=128\n            or config.get(\'preprocessing\')!=\'EXIF orientation, RGB, bilinear resize128x128, pixels/255\'):\n        raise ValueError(\'Kiến trúc/preprocessing CNN không khớp baseline.\')\n    if name==\'rf\':\n        model=joblib.load(folder/\'model.joblib\')\n        for parameter in [\'n_estimators\',\'max_depth\',\'min_samples_leaf\',\'max_features\']:\n            if model.get_params()[parameter]!=config.get(parameter):raise ValueError(\'RF params khác config.\')\n        model.set_params(n_jobs=1)\n        return model,None,config\n    model=GestureMLP() if name==\'mlp\' else GestureCNN()\n    model.load_state_dict(torch.load(folder/\'model.pt\',map_location=\'cpu\',weights_only=True));model.eval()\n    scaler=joblib.load(folder/\'scaler.joblib\') if name==\'mlp\' else None\n    return model,scaler,config\n\n\ndef benchmark_models(results,test_frame,data_root,seed=42,warmup=20,sample_count=200):\n    torch.set_num_threads(1)\n    ordered=test_frame.sort_values(\'sample_id\')\n    benchmark=ordered.sample(min(sample_count,len(ordered)),random_state=seed).reset_index(drop=True)\n    features=benchmark[LM_COLUMNS].to_numpy(dtype=np.float64)\n    tensors=[]\n    for path in benchmark.image_path:\n        with Image.open(Path(data_root)/path) as image:tensors.append(image_tensor(image,False,128).unsqueeze(0))\n    rows=[]\n    for name,result in results.items():\n        model,scaler,config=load_saved_model(name,result[\'directory\'])\n        if name==\'cnn\' and config.get(\'image_size\')!=128:raise ValueError(\'Benchmark CNN yêu cầu preprocessing128 như baseline.\')\n        def predict(index):\n            if name==\'rf\':return model.predict_proba(features[index:index+1])\n            with torch.inference_mode():\n                if name==\'mlp\':\n                    values=scaler.transform(features[index:index+1]).astype(np.float32)\n                    return model(torch.from_numpy(values)).softmax(dim=1)\n                return model(tensors[index]).softmax(dim=1)\n        for i in range(warmup):predict(i%len(benchmark))\n        timings=[]\n        for i in range(len(benchmark)):\n            start=time.perf_counter_ns();predict(i);timings.append((time.perf_counter_ns()-start)/1e6)\n        rows.append({\'model\':name,\'cpu_latency_median_ms\':float(np.median(timings)),\n                     \'cpu_latency_p95_ms\':float(np.percentile(timings,95)),\'benchmark_samples\':len(benchmark),\n                     \'batch_size\':1,\'threads\':1,\'warmup\':warmup})\n    return pd.DataFrame(rows)\n'
exec(compile(_source, 'scripts/kaggle_notebook_support.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: handgesture/models.py
_module = types.ModuleType('handgesture.models')
_module.__file__ = 'handgesture/models.py'
sys.modules['handgesture.models'] = _module
_source = '"""Model registry and inference; inference never downloads pretrained weights."""\nfrom pathlib import Path\nimport json\nimport random\n\nimport joblib\nimport numpy as np\nfrom PIL import Image,ImageEnhance,ImageOps\nimport torch\nfrom torch import nn\n\nfrom handgesture.data import MAPPING,LM_COLUMNS\nfrom handgesture.preprocessing import sha256\n\nARCHITECTURES = [\'gap\',\'spatial\',\'mobilenet_v3_small\']\nIMAGENET_MEAN = torch.tensor([.485,.456,.406])[:,None,None]\nIMAGENET_STD = torch.tensor([.229,.224,.225])[:,None,None]\n\n\nclass GestureMLP(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.network = nn.Sequential(nn.Linear(63,128),nn.ReLU(),nn.Dropout(.2),\n                                     nn.Linear(128,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,6))\n    def forward(self,x): return self.network(x)\n\n\nclass GestureCNN(nn.Module):\n    def __init__(self,spatial=False):\n        super().__init__()\n        blocks=[];channels=3\n        for output in [32,64,128]:\n            blocks += [nn.Conv2d(channels,output,3,padding=1),nn.BatchNorm2d(output),\n                       nn.ReLU(),nn.MaxPool2d(2)]\n            channels=output\n        self.features=nn.Sequential(*blocks)\n        self.classifier=(nn.Sequential(nn.AdaptiveAvgPool2d((4,4)),nn.Flatten(),\n                         nn.Linear(2048,128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,6)) if spatial else\n                         nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.2),nn.Linear(128,6)))\n    def forward(self,x): return self.classifier(self.features(x))\n\n\ndef build_model(architecture,initialize_pretrained=False,weights_path=None):\n    if architecture==\'mlp\': return GestureMLP()\n    if architecture in {\'gap\',\'spatial\'}: return GestureCNN(architecture==\'spatial\')\n    if architecture!=\'mobilenet_v3_small\': raise ValueError(f\'Unknown architecture: {architecture}\')\n    from torchvision.models import mobilenet_v3_small\n    model=mobilenet_v3_small(weights=None)\n    if initialize_pretrained:\n        if weights_path is None or not Path(weights_path).is_file():\n            raise ValueError(\'Pretrained MobileNet weights file is required; no random-init fallback\')\n        model.load_state_dict(torch.load(weights_path,map_location=\'cpu\',weights_only=True))\n    model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,6)\n    return model\n\n\ndef image_tensor(image,architecture=\'gap\',training=False):\n    image=Image.fromarray(image) if isinstance(image,np.ndarray) else image.convert(\'RGB\')\n    image=ImageOps.exif_transpose(image).convert(\'RGB\')\n    size=224 if architecture==\'mobilenet_v3_small\' else 128\n    image=image.resize((size,size),Image.Resampling.BILINEAR)\n    if training:\n        image=image.rotate(random.uniform(-10,10),resample=Image.Resampling.BILINEAR)\n        image=ImageEnhance.Brightness(image).enhance(random.uniform(.85,1.15))\n        image=ImageEnhance.Contrast(image).enhance(random.uniform(.85,1.15))\n    tensor=torch.from_numpy(np.array(image,dtype=np.float32).transpose(2,0,1).copy())/255\n    if architecture==\'mobilenet_v3_small\': tensor=(tensor-IMAGENET_MEAN)/IMAGENET_STD\n    return tensor\n\n\nclass GesturePredictor:\n    def __init__(self,directory,preprocessing_config=None,allow_legacy=False):\n        self.directory=Path(directory)\n        self.config=json.loads((self.directory/\'model_config.json\').read_text())\n        self.kind=self.config[\'model\']\n        fingerprint=self.config.get(\'preprocessing_fingerprint\')\n        if fingerprint is None:\n            if not allow_legacy: raise ValueError(\'Legacy preprocessing is unverified; use a v2 checkpoint for webcam\')\n            from scripts.kaggle_notebook_support import load_saved_model\n            self.model,self.scaler,_=load_saved_model(self.kind,self.directory)\n            self.architecture=\'gap\' if self.kind==\'cnn\' else self.kind\n            return\n        if preprocessing_config is not None: preprocessing_config.assert_matches(fingerprint)\n        if self.config.get(\'class_mapping\')!=MAPPING:\n            raise ValueError(\'Checkpoint class mapping mismatch\')\n        record_path=self.directory/(\'metrics.json\' if (self.directory/\'metrics.json\').exists() else \'run.json\')\n        record=json.loads(record_path.read_text())\n        for filename,digest in record[\'artifact_sha256\'].items():\n            if sha256(self.directory/filename)!=digest: raise ValueError(f\'Artifact hash mismatch: {filename}\')\n        if record[\'config\']!=self.config: raise ValueError(\'Config differs from run record\')\n        self.architecture=self.config[\'architecture\']\n        if self.kind==\'rf\':\n            self.model=joblib.load(self.directory/\'model.joblib\');self.model.set_params(n_jobs=1)\n        else:\n            self.model=build_model(self.architecture)\n            self.model.load_state_dict(torch.load(self.directory/\'model.pt\',map_location=\'cpu\',weights_only=True))\n            self.model.eval()\n        self.scaler=joblib.load(self.directory/\'scaler.joblib\') if self.kind==\'mlp\' else None\n\n    def predict(self,features=None,roi=None):\n        return self.predict_input(self.prepare_input(features,roi))\n\n    def prepare_input(self,features=None,roi=None):\n        if self.kind in {\'rf\',\'mlp\'}:\n            return np.asarray(features,dtype=float).reshape(-1,63)\n        return image_tensor(roi,self.architecture).unsqueeze(0)\n\n    def predict_input(self,value):\n        if self.kind==\'rf\': return self.model.predict_proba(value)[0]\n        with torch.inference_mode():\n            if self.kind==\'mlp\':\n                x=self.scaler.transform(value).astype(np.float32)\n                tensor=torch.from_numpy(x)\n            else: tensor=value\n            return self.model(tensor).softmax(1).numpy()[0]\n'
exec(compile(_source, 'handgesture/models.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: handgesture/training.py
_module = types.ModuleType('handgesture.training')
_module.__file__ = 'handgesture/training.py'
sys.modules['handgesture.training'] = _module
_source = '"""Train on train/validation only. Test evaluation is a separate finalization step."""\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport copy\nimport json\nimport os\nimport random\nimport shutil\nimport time\nimport warnings\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.preprocessing import StandardScaler\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader, TensorDataset\n\nfrom handgesture.data import load_dataset, write_json, MAPPING, GESTURES, LM_COLUMNS\nfrom handgesture.models import build_model, image_tensor, GesturePredictor\nfrom handgesture.preprocessing import sha256\nfrom handgesture.runtime import progress, validate_workers, require_cuda\nfrom scripts import kaggle_notebook_support as reporting\n\n\nclass ROIImages(Dataset):\n    def __init__(self,frame,root,architecture,training=False):\n        self.frame=frame.reset_index(drop=True);self.root=Path(root)\n        self.architecture=architecture;self.training=training\n    def __len__(self): return len(self.frame)\n    def __getitem__(self,index):\n        row=self.frame.iloc[index]\n        with Image.open(self.root/row.image_path) as image:\n            x=image_tensor(image,self.architecture,self.training)\n        return x,int(row.class_id)\n\n\ndef environment(device):\n    value=reporting.environment_info(device)\n    import importlib.metadata\n    for name in [\'torch\',\'torchvision\',\'mediapipe\']:\n        try:value[\'versions\'][name]=importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:pass\n    value[\'gpu\']=torch.cuda.get_device_name() if str(device).startswith(\'cuda\') else None\n    return value\n\n\ndef prediction_frame(frame,probabilities):\n    p=np.asarray(probabilities)\n    if p.shape!=(len(frame),6) or not np.isfinite(p).all() or not np.allclose(p.sum(1),1,atol=1e-5):\n        raise ValueError(\'Invalid probability matrix\')\n    result=frame[[\'sample_id\',\'class_id\',\'image_path\']].rename(columns={\'class_id\':\'y_true\'}).copy()\n    result[\'y_pred\']=p.argmax(1)\n    for i in range(6):result[f\'prob_{i}\']=p[:,i]\n    return result\n\n\ndef pass_epoch(model,loader,device,optimizer=None,freeze_backbone=False,progress_info=None):\n    started=time.monotonic()\n    model.train(optimizer is not None)\n    if freeze_backbone: model.features.eval()\n    losses=0.;prob=[];truth=[]\n    with torch.set_grad_enabled(optimizer is not None):\n        for batch_index,(x,y) in enumerate(loader,1):\n            x,y=x.to(device),y.to(device)\n            if optimizer is not None:optimizer.zero_grad(set_to_none=True)\n            logits=model(x);loss=nn.functional.cross_entropy(logits,y)\n            if optimizer is not None:loss.backward();optimizer.step()\n            losses+=float(loss.detach().cpu())*len(y)\n            prob.append(logits.detach().softmax(1).cpu().numpy());truth.extend(y.cpu().tolist())\n            if progress_info and (batch_index==1 or batch_index%25==0 or batch_index==len(loader)):\n                progress(**progress_info, completed=batch_index, total=len(loader),\n                         elapsed_seconds=time.monotonic()-started)\n    values=np.concatenate(prob)\n    return losses/len(truth),reporting.classification_metrics(truth,values),values\n\n\ndef make_loader(dataset,batch,shuffle,seed,workers):\n    validate_workers(workers)\n    kwargs={\'multiprocessing_context\':\'fork\'} if workers else {}\n    return DataLoader(dataset,batch_size=batch,shuffle=shuffle,num_workers=workers,\n                      generator=torch.Generator().manual_seed(seed),\n                      worker_init_fn=reporting.seed_worker,**kwargs)\n\n\ndef train_run(data_root,output_dir,kind,architecture=None,weights_path=None,\n              smoke=False,device=\'auto\',workers=0,seed=42,max_epochs=None):\n    """No test image, feature, label or probability enters training/selection."""\n    validate_workers(workers)\n    if kind!=\'rf\' and os.environ.get(\'HG_REQUIRE_CUDA\')==\'1\':\n        if device not in {\'auto\',\'cuda\'}:raise RuntimeError(\'CUDA device required on Kaggle\')\n        require_cuda()\n    if kind not in {\'rf\',\'mlp\',\'cnn\'}: raise ValueError(\'Unknown model\')\n    architecture=architecture or (\'gap\' if kind==\'cnn\' else kind)\n    if kind!=\'cnn\' and architecture!=kind:raise ValueError(\'Architecture does not match model kind\')\n    if kind==\'cnn\' and architecture not in {\'gap\',\'spatial\',\'mobilenet_v3_small\'}:\n        raise ValueError(\'Unknown CNN architecture\')\n    progress(stage=\'training_setup\',model=architecture,status=\'running\',completed=0,total=None)\n    frame,preparation=load_dataset(data_root)\n    smoke=bool(smoke or preparation.get(\'smoke_test\',False))\n    splits={name:frame.loc[frame.split.eq(name)].copy() for name in [\'train\',\'validation\']}\n    if smoke:\n        splits={name:rows.groupby(\'class_id\',group_keys=False).head(8 if name==\'train\' else 4)\n                for name,rows in splits.items()}\n    output_dir=Path(output_dir)\n    if output_dir.exists() and any(output_dir.iterdir()):\n        raise ValueError(f\'Refusing to overwrite run: {output_dir}\')\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    device=torch.device(\'cuda\' if device==\'auto\' and torch.cuda.is_available() else \'cpu\' if device==\'auto\' else device)\n    if kind==\'rf\':device=torch.device(\'cpu\')\n    mobile=architecture==\'mobilenet_v3_small\'\n    official_weights=\'047dcff4addef86ea5bc2eff13c9614dc11f47ab1160d0a71a25e7db994f4e1f\'\n    if mobile and weights_path is not None and sha256(weights_path)!=official_weights and not smoke:\n        raise ValueError(\'Pretrained weights hash is not official IMAGENET1K_V1\')\n    # Build pretrained model before creating any output, and never silently fall back.\n    model=None if kind==\'rf\' else build_model(architecture,initialize_pretrained=mobile,weights_path=weights_path)\n    output_dir.mkdir(parents=True,exist_ok=True)\n    fingerprint=preparation[\'preprocessing_fingerprint\']\n    protocol={\'manifest_sha256\':sha256(Path(data_root)/\'metadata/manifest.csv\'),\n              \'preprocessing_fingerprint\':fingerprint,\'class_mapping\':MAPPING,\'seed\':seed,\n              \'smoke_test\':smoke,\'sample_counts\':{**frame.split.value_counts().to_dict(),\n                      **{name:len(rows) for name,rows in splits.items()}},\n              \'validation_sample_ids\':splits[\'validation\'].sample_id.tolist(),\n              \'limits\':preparation[\'limits\']}\n    config={\'model\':kind,\'architecture\':architecture,\'class_mapping\':MAPPING,\'features\':LM_COLUMNS,\n            \'seed\':seed,\'smoke_test\':smoke,\'run_name\':output_dir.name,\n            \'preprocessing\':preparation[\'preprocessing\'],\'preprocessing_fingerprint\':fingerprint,\n            \'image_size\':224 if mobile else 128,\'dropout\':.3 if architecture==\'spatial\' else .2,\n            \'optimizer\':\'AdamW\',\'weight_decay\':1e-4,\n            \'train_augmentation\':{\'rotation_degrees\':10,\'brightness\':[.85,1.15],\n                                  \'contrast\':[.85,1.15],\'flip\':False},\n            \'selection\':\'validation macro-F1 descending, validation loss ascending\'}\n    if kind!=\'cnn\':\n        config.update(image_size=None,train_augmentation=None)\n    if kind==\'rf\':\n        config.update(dropout=None,optimizer=None,weight_decay=None)\n    if mobile:\n        config[\'pretrained\']={\'name\':\'IMAGENET1K_V1\' if sha256(weights_path)==official_weights else \'TEST_FIXTURE_NOT_PRETRAINED\',\n              \'sha256\':sha256(weights_path),\n              \'url\':\'https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth\',\n              \'mean\':[.485,.456,.406],\'std\':[.229,.224,.225]}\n    history=[];start=time.perf_counter();training_time=0.;best_loss=None\n    if kind==\'rf\':\n        options={\'n_estimators\':10 if smoke else 300,\'max_depth\':20,\'min_samples_leaf\':2,\n                 \'max_features\':\'sqrt\',\'n_jobs\':1 if smoke else -1,\'random_state\':seed}\n        config.update(options)\n        model=RandomForestClassifier(**options)\n        progress(stage=\'training\',model=\'rf\',status=\'running\',completed=0,total=1)\n        t=time.perf_counter();model.fit(splits[\'train\'][LM_COLUMNS].to_numpy(),splits[\'train\'].class_id)\n        training_time=time.perf_counter()-t\n        train_metrics=reporting.classification_metrics(splits[\'train\'].class_id,model.predict_proba(splits[\'train\'][LM_COLUMNS].to_numpy()))\n        probabilities=model.predict_proba(splits[\'validation\'][LM_COLUMNS].to_numpy())\n        validation=reporting.classification_metrics(splits[\'validation\'].class_id,probabilities)\n        history=[{\'step\':1,\'train_accuracy\':train_metrics[\'accuracy\'],\'train_macro_f1\':train_metrics[\'macro_f1\'],\n                  \'validation_accuracy\':validation[\'accuracy\'],\'validation_macro_f1\':validation[\'macro_f1\']}]\n        joblib.dump(model,output_dir/\'model.joblib\')\n    else:\n        batch=256 if kind==\'mlp\' else 64\n        if kind==\'mlp\':\n            scaler=StandardScaler().fit(splits[\'train\'][LM_COLUMNS].to_numpy())\n            joblib.dump(scaler,output_dir/\'scaler.joblib\')\n            datasets={name:TensorDataset(torch.from_numpy(scaler.transform(rows[LM_COLUMNS].to_numpy()).astype(np.float32)),\n                          torch.tensor(rows.class_id.to_numpy(),dtype=torch.long)) for name,rows in splits.items()}\n        else:\n            datasets={name:ROIImages(rows,data_root,architecture,name==\'train\') for name,rows in splits.items()}\n        loaders={name:make_loader(ds,batch,name==\'train\',seed,workers) for name,ds in datasets.items()}\n        model=model.to(device)\n        epochs=2 if smoke else (max_epochs or (60 if kind==\'mlp\' or not mobile else 40))\n        patience=8 if kind==\'mlp\' or mobile else 12\n        warmup=1 if smoke and mobile else 5 if mobile else 0\n        lr=1e-3 if kind==\'mlp\' or mobile else 3e-4\n        if mobile:\n            for p in model.features.parameters():p.requires_grad_(False)\n            optimizer=torch.optim.AdamW(model.classifier.parameters(),lr=1e-3,weight_decay=1e-4)\n        else:optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)\n        scheduler=None if kind==\'mlp\' or mobile else torch.optim.lr_scheduler.ReduceLROnPlateau(\n              optimizer,mode=\'max\',factor=.5,patience=3,threshold=1e-4,threshold_mode=\'abs\',min_lr=1e-6)\n        config.update(batch_size=batch,max_epochs=epochs,patience=patience,learning_rate=lr,\n                      scheduler=None if scheduler is None else {\'name\':\'ReduceLROnPlateau\',\'mode\':\'max\',\'factor\':.5,\n                           \'patience\':3,\'threshold\':1e-4,\'threshold_mode\':\'abs\',\'min_lr\':1e-6},\n                      transfer_schedule={\'frozen_epochs\':warmup,\'backbone_lr\':1e-5,\'classifier_lr\':1e-4} if mobile else None)\n        best=(-float(\'inf\'),float(\'inf\'));best_state=None;bad=0\n        for epoch in range(1,epochs+1):\n            epoch_started=time.monotonic()\n            info={\'model\':architecture,\'epoch\':epoch,\'max_epochs\':epochs,\'status\':\'running\'}\n            if mobile and epoch==warmup+1:\n                for p in model.features.parameters():p.requires_grad_(True)\n                optimizer=torch.optim.AdamW([{\'params\':model.features.parameters(),\'lr\':1e-5},\n                         {\'params\':model.classifier.parameters(),\'lr\':1e-4}],weight_decay=1e-4)\n                bad=0\n            rates=[group[\'lr\'] for group in optimizer.param_groups]\n            t=time.perf_counter()\n            train_loss,train_metrics,_=pass_epoch(model,loaders[\'train\'],device,optimizer,mobile and epoch<=warmup,\n                progress_info={**info,\'stage\':\'train_batches\'})\n            training_time+=time.perf_counter()-t\n            loss,val,_=pass_epoch(model,loaders[\'validation\'],device,\n                progress_info={**info,\'stage\':\'validation_batches\'})\n            history.append({\'step\':epoch,\'epoch\':epoch,\'learning_rate\':rates[-1],\n                 \'backbone_learning_rate\':0 if mobile and epoch<=warmup else rates[0],\n                 \'phase\':\'frozen\' if mobile and epoch<=warmup else \'finetune\' if mobile else \'train\',\n                 \'train_loss\':train_loss,\'validation_loss\':loss,\'train_accuracy\':train_metrics[\'accuracy\'],\n                 \'validation_accuracy\':val[\'accuracy\'],\'train_macro_f1\':train_metrics[\'macro_f1\'],\n                 \'validation_macro_f1\':val[\'macro_f1\'],\'smoke_test\':smoke})\n            progress(stage=\'epoch\',model=architecture,epoch=epoch,max_epochs=epochs,\n                     validation_macro_f1=val[\'macro_f1\'],validation_loss=loss,\n                     elapsed_seconds=time.monotonic()-epoch_started)\n            print(f\'{output_dir.name} epoch {epoch}/{epochs}: val F1={val["macro_f1"]:.5f}, loss={loss:.5f}\',flush=True)\n            if val[\'macro_f1\']>best[0] or (val[\'macro_f1\']==best[0] and loss<best[1]):\n                best=(val[\'macro_f1\'],loss);best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n                config[\'best_epoch\']=epoch;bad=0\n            else:bad+=1\n            if scheduler:scheduler.step(val[\'macro_f1\'])\n            if bad>=patience and (not mobile or epoch>warmup):break\n        model.load_state_dict(best_state);model.eval()\n        torch.save(best_state,output_dir/\'model.pt\')\n        best_loss,validation,probabilities=pass_epoch(model,loaders[\'validation\'],device)\n    wall=time.perf_counter()-start\n    write_json(output_dir/\'model_config.json\',config);write_json(output_dir/\'class_mapping.json\',MAPPING)\n    hist=pd.DataFrame(history);hist[\'smoke_test\']=smoke\n    for col in reporting.HISTORY_COLUMNS:\n        if col not in hist:hist[col]=np.nan\n    hist.to_csv(output_dir/\'history.csv\',index=False)\n    prediction_frame(splits[\'validation\'],probabilities).to_csv(output_dir/\'validation_predictions.csv\',index=False)\n    reporting.show_history(hist,output_dir/\'history.png\',smoke,kind)\n    artifacts=reporting.MODEL_ARTIFACTS[kind]\n    record={\'model\':kind,\'config\':config,\'protocol\':protocol,\'validation_metrics\':validation,\n            \'validation_loss\':best_loss,\'training_time_s\':training_time,\'fit_wall_time_s\':wall,\n            \'environment\':environment(device),\'artifact_sha256\':{f:sha256(output_dir/f) for f in artifacts},\n            \'validation_predictions_sha256\':sha256(output_dir/\'validation_predictions.csv\'),\n            \'history_sha256\':sha256(output_dir/\'history.csv\')}\n    write_json(output_dir/\'run.json\',record)\n    return record\n\n\ndef read_run(directory):\n    directory=Path(directory)\n    record=json.loads((directory/\'run.json\').read_text())\n    for f,digest in {**record[\'artifact_sha256\'],\'validation_predictions.csv\':record[\'validation_predictions_sha256\'],\n                     \'history.csv\':record[\'history_sha256\']}.items():\n        if sha256(directory/f)!=digest:raise ValueError(f\'Run artifact hash mismatch: {f}\')\n    if json.loads((directory/\'model_config.json\').read_text())!=record[\'config\']:\n        raise ValueError(\'Run config mismatch\')\n    p=pd.read_csv(directory/\'validation_predictions.csv\')\n    validate_predictions(p)\n    if set(p.sample_id)!=set(record[\'protocol\'][\'validation_sample_ids\']):raise ValueError(\'Validation IDs mismatch\')\n    measured=reporting.classification_metrics(p.y_true,p[[f\'prob_{i}\' for i in range(6)]].to_numpy())\n    for key in reporting.METRIC_COLUMNS:\n        if not np.isclose(measured[key],record[\'validation_metrics\'][key],rtol=1e-9,atol=1e-12):\n            raise ValueError(\'Validation metric mismatch\')\n    return {**record,\'directory\':str(directory)}\n\n\ndef validate_predictions(predictions):\n    if not predictions.sample_id.is_unique or not set(predictions.y_true).issubset(range(6)):\n        raise ValueError(\'Invalid prediction labels/IDs\')\n    p=predictions[[f\'prob_{i}\' for i in range(6)]].to_numpy()\n    if (not np.isfinite(p).all() or (p<0).any() or (p>1).any()\n        or not np.allclose(p.sum(1),1,atol=1e-5) or not np.array_equal(p.argmax(1),predictions.y_pred)):\n        raise ValueError(\'Invalid probabilities/predictions\')\n\n\ndef protocol_identity(record):\n    p=record[\'protocol\']\n    return tuple(json.dumps(p.get(k),sort_keys=True) for k in [\'manifest_sha256\',\'preprocessing_fingerprint\',\n                                    \'smoke_test\',\'seed\',\'class_mapping\',\'sample_counts\',\'validation_sample_ids\'])\n\n\ndef choose_cnn(records):\n    if not records or any(r[\'model\']!=\'cnn\' for r in records):raise ValueError(\'Need CNN candidates\')\n    if len({protocol_identity(r) for r in records})!=1:raise ValueError(\'Mixed candidate protocol\')\n    return sorted(records,key=lambda r:(-r[\'validation_metrics\'][\'macro_f1\'],\n                       float(\'inf\') if r[\'validation_loss\'] is None else r[\'validation_loss\'],r[\'directory\']))[0]\n\n\ndef require_cnn_candidates(records):\n    if (len(records)!=3 or len({str(Path(r[\'directory\']).resolve()) for r in records})!=3\n            or any(r[\'model\']!=\'cnn\' for r in records)\n            or {r[\'config\'][\'architecture\'] for r in records}!={\'gap\',\'spatial\',\'mobilenet_v3_small\'}):\n        raise ValueError(\'Need exactly three distinct CNN candidates: gap, spatial, mobilenet_v3_small\')\n    mobile=next(r for r in records if r[\'config\'][\'architecture\']==\'mobilenet_v3_small\')\n    if not mobile[\'protocol\'][\'smoke_test\']:\n        weights=mobile[\'config\'].get(\'pretrained\',{})\n        if weights.get(\'name\')!=\'IMAGENET1K_V1\' or weights.get(\'sha256\')!=\'047dcff4addef86ea5bc2eff13c9614dc11f47ab1160d0a71a25e7db994f4e1f\':\n            raise ValueError(\'MobileNet candidate lacks official pretrained provenance\')\n\n\ndef predict_dataset(predictor,frame,data_root):\n    if predictor.kind==\'rf\':return predictor.model.predict_proba(frame[LM_COLUMNS].to_numpy())\n    if predictor.kind==\'mlp\':\n        values=predictor.scaler.transform(frame[LM_COLUMNS].to_numpy()).astype(np.float32)\n        dataset=TensorDataset(torch.from_numpy(values),torch.tensor(frame.class_id.to_numpy()))\n    else:dataset=ROIImages(frame,data_root,predictor.architecture)\n    return pass_epoch(predictor.model,DataLoader(dataset,batch_size=64),\'cpu\')[2]\n\n\ndef finalize(data_root,rf_dir,mlp_dir,cnn_dirs,output_root):\n    records=[read_run(rf_dir),read_run(mlp_dir)]+[read_run(p) for p in cnn_dirs]\n    require_cnn_candidates(records[2:])\n    if len({protocol_identity(r) for r in records})!=1:raise ValueError(\'Mixed experiment protocol\')\n    if [r[\'model\'] for r in records[:2]]!=[\'rf\',\'mlp\']:raise ValueError(\'Wrong RF/MLP runs\')\n    frame,preparation=load_dataset(data_root)\n    reference=records[0][\'protocol\']\n    if reference[\'manifest_sha256\']!=sha256(Path(data_root)/\'metadata/manifest.csv\'):\n        raise ValueError(\'Dataset manifest differs from training\')\n    if reference[\'preprocessing_fingerprint\']!=preparation[\'preprocessing_fingerprint\']:\n        raise ValueError(\'Dataset preprocessing differs from training\')\n    # Validate validation IDs/truth against original dataset before selection.\n    validation=frame.loc[frame.split.eq(\'validation\')].set_index(\'sample_id\')\n    for r in records:\n        p=pd.read_csv(Path(r[\'directory\'])/\'validation_predictions.csv\').set_index(\'sample_id\')\n        if (not set(p.index).issubset(validation.index)\n                or (not reference[\'smoke_test\'] and set(p.index)!=set(validation.index))\n                or not np.array_equal(p.y_true,validation.loc[p.index].class_id)):\n            raise ValueError(\'Validation truth differs from dataset\')\n    chosen=choose_cnn(records[2:]);selected=[*records[:2],chosen]\n    output_root=Path(output_root)\n    if output_root.exists() and any(output_root.iterdir()):raise ValueError(\'Final output exists; use a new experiment directory\')\n    output_root.mkdir(parents=True,exist_ok=True)\n    write_json(output_root/\'selection.json\',{\'criterion\':\'validation macro-F1, then loss; test not used\',\n           \'cnn_candidates\':[{\'directory\':r[\'directory\'],\'validation_macro_f1\':r[\'validation_metrics\'][\'macro_f1\'],\n                              \'validation_loss\':r[\'validation_loss\'],\'checkpoint_sha256\':r[\'artifact_sha256\'][\'model.pt\']} for r in records[2:]],\n           \'selected_cnn\':chosen[\'directory\'],\'protocol\':reference,\'smoke_test\':reference[\'smoke_test\']})\n    test=frame.loc[frame.split.eq(\'test\')].copy()\n    if reference[\'smoke_test\']:test=test.groupby(\'class_id\',group_keys=False).head(4)\n    protocol={**reference,\'test_sample_ids\':test.sample_id.tolist(),\n              \'sample_counts\':{**reference[\'sample_counts\'],\'test\':len(test)}}\n    for r in selected:\n        kind=r[\'model\'];folder=output_root/kind;shutil.copytree(r[\'directory\'],folder)\n        predictor=GesturePredictor(folder)\n        probabilities=predict_dataset(predictor,test,data_root)\n        original_history=(folder/\'history.csv\').read_bytes()\n        metrics,predictions=reporting.export_evaluation(folder,kind,test,probabilities,protocol,r[\'environment\'],\n                  r[\'training_time_s\'],r[\'fit_wall_time_s\'],r[\'validation_metrics\'],r[\'config\'])\n        # Legacy plot/table generation standardizes the CSV. Preserve all measured\n        # v2 columns (transfer phase and backbone LR) and the original run hash.\n        (folder/\'history.csv\').write_bytes(original_history)\n        metrics[\'validation_predictions_sha256\']=sha256(folder/\'validation_predictions.csv\')\n        metrics[\'preprocessing_fingerprint\']=reference[\'preprocessing_fingerprint\']\n        write_json(folder/\'metrics.json\',metrics)\n        reporting.show_errors(predictions,data_root,folder/\'misclassified_gallery.png\',smoke=reference[\'smoke_test\'])\n        # Three most common directed confusions, up to four examples per pair.\n        errors=predictions.loc[predictions.y_true.ne(predictions.y_pred)]\n        top=errors.groupby([\'y_true\',\'y_pred\']).size().sort_values(ascending=False).head(3)\n        for index,(pair,_) in enumerate(top.items(),1):\n            examples=errors.loc[errors.y_true.eq(pair[0]) & errors.y_pred.eq(pair[1])].head(4)\n            reporting.show_errors(examples,data_root,folder/f\'confusion_pair_{index}.png\',limit=4,smoke=reference[\'smoke_test\'])\n        reporting.write_model_report(folder,metrics,predictions)\n        pairs=sorted(folder.glob(\'confusion_pair_*.png\'))\n        if pairs:\n            with (folder/\'report.md\').open(\'a\') as report:\n                report.write(\'\\n## Các cặp nhầm lẫn nhiều nhất\\n\\n\')\n                for path in pairs:report.write(f\'[{path.stem}]({path.name})\\n\\n\')\n    results=verify_outputs(output_root,data_root)\n    reporting.export_comparison(results,output_root/\'comparison\')\n    return results\n\n\ndef verify_outputs(root,data_root=None,verify_derived=True):\n    root=Path(root)\n    results=reporting.load_comparison({n:root/n for n in [\'rf\',\'mlp\',\'cnn\']})\n    dataset=None\n    if data_root is not None:\n        dataset,preparation=load_dataset(data_root)\n        if results[\'rf\'][\'metrics\'][\'protocol\'][\'manifest_sha256\']!=sha256(Path(data_root)/\'metadata/manifest.csv\'):\n            raise ValueError(\'Dataset manifest mismatch\')\n        if results[\'rf\'][\'metrics\'][\'protocol\'][\'preprocessing_fingerprint\']!=preparation[\'preprocessing_fingerprint\']:\n            raise ValueError(\'Dataset preprocessing fingerprint mismatch\')\n    for name,result in results.items():\n        metrics=result[\'metrics\'];folder=result[\'directory\'];p=result[\'predictions\']\n        run=read_run(folder)\n        if run[\'config\']!=metrics[\'config\']:raise ValueError(\'Final config differs from selected run\')\n        if (metrics[\'config\'][\'preprocessing_fingerprint\']!=metrics[\'protocol\'][\'preprocessing_fingerprint\']\n                or metrics[\'protocol\'][\'preprocessing_fingerprint\']!=results[\'rf\'][\'metrics\'][\'protocol\'][\'preprocessing_fingerprint\']):\n            raise ValueError(\'Mixed preprocessing fingerprint\')\n        reporting.validate_result_metrics(metrics,p)\n        if verify_derived:\n            expected_summary,expected_classes=reporting.evaluation_tables(metrics)\n            for filename,expected in [(\'summary.csv\',expected_summary),(\'per_class_metrics.csv\',expected_classes)]:\n                try:\n                    pd.testing.assert_frame_equal(pd.read_csv(folder/filename),expected,check_dtype=False,\n                                                  rtol=1e-9,atol=1e-12)\n                except AssertionError as exc:raise ValueError(f\'Derived output mismatch: {filename}\') from exc\n            cm=reporting.confusion_matrix(p.y_true,p.y_pred,labels=range(6))\n            for filename,expected in [(\'confusion_matrix.csv\',cm),\n                                     (\'confusion_matrix_normalized.csv\',cm/np.maximum(cm.sum(1,keepdims=True),1))]:\n                if not np.allclose(pd.read_csv(folder/filename,index_col=0).to_numpy(),expected,rtol=1e-9,atol=1e-12):\n                    raise ValueError(f\'Derived output mismatch: {filename}\')\n            for filename in [\'report.md\',\'history.png\',\'confusion_matrix.png\']:\n                if not (folder/filename).is_file():raise ValueError(f\'Missing output: {filename}\')\n        if sha256(folder/\'validation_predictions.csv\')!=metrics[\'validation_predictions_sha256\']:\n            raise ValueError(\'Validation prediction hash mismatch\')\n        val=pd.read_csv(folder/\'validation_predictions.csv\');validate_predictions(val)\n        measured=reporting.classification_metrics(val.y_true,val[[f\'prob_{i}\' for i in range(6)]].to_numpy())\n        for k in reporting.METRIC_COLUMNS:\n            if not np.isclose(metrics[\'validation_metrics\'][k],measured[k],rtol=1e-9,atol=1e-12):\n                raise ValueError(\'Validation metric mismatch\')\n        if dataset is not None:\n            for split,pred in [(\'test\',p),(\'validation\',val)]:\n                truth=dataset.loc[dataset.split.eq(split)].set_index(\'sample_id\')\n                if (not set(pred.sample_id).issubset(truth.index)\n                    or (not metrics[\'protocol\'][\'smoke_test\'] and set(pred.sample_id)!=set(truth.index))\n                    or not np.array_equal(pred.y_true,truth.loc[pred.sample_id].class_id)):\n                    raise ValueError(\'Prediction truth/IDs differ from dataset\')\n    return results\n\n\ndef regenerate_outputs(root,data_root):\n    """Repair derived reports after validating all measured artifacts, without fitting."""\n    results=verify_outputs(root,data_root,verify_derived=False)\n    for result in results.values():\n        folder=result[\'directory\'];metrics=result[\'metrics\'];predictions=result[\'predictions\']\n        history=(folder/\'history.csv\').read_bytes()\n        try:\n            reporting.show_errors(predictions,data_root,folder/\'misclassified_gallery.png\',\n                                  smoke=metrics[\'protocol\'][\'smoke_test\'])\n            reporting.write_model_outputs(folder,metrics,predictions)\n            pairs=sorted(folder.glob(\'confusion_pair_*.png\'))\n            if pairs:\n                with (folder/\'report.md\').open(\'a\') as report:\n                    report.write(\'\\n## Các cặp nhầm lẫn nhiều nhất\\n\\n\')\n                    for path in pairs:report.write(f\'[{path.stem}]({path.name})\\n\\n\')\n        finally:\n            (folder/\'history.csv\').write_bytes(history)\n    reporting.export_comparison(results,Path(root)/\'comparison\')\n    return verify_outputs(root,data_root)\n'
exec(compile(_source, 'handgesture/training.py', 'exec'), _module.__dict__)

In [ ]:
from handgesture.training import train_run
from handgesture.runtime import stage
DATA_ROOT = Path(DATA_ROOT) if DATA_ROOT else find_dataset(True)
with stage('training',model='mlp'):
    train_run(DATA_ROOT, OUTPUT_ROOT/'runs'/'mlp', 'mlp', smoke=SMOKE_TEST, device=DEVICE, workers=NUM_WORKERS)
